# Семантическая классификация пользовательских запросов

Аттестационная работа по программе профессиональной переподготовки «Аналитика данных PRO DATA». Официальная тема: «Кластеризация пользовательских запросов по семантике PRO DA (FarPost)».

Практическая задача текущей реализации — определить наличие пяти категорий в тексте: занятость, профессия, дополнительный признак, условия и общая фраза. Один запрос может содержать несколько категорий: «охранник вахтой без опыта» содержит профессию, формат занятости и дополнительный признак. Названия конкретных профессий и условий служат исходной разметкой, а ML-модели предсказывают присутствие категорий. Алгоритмы кластеризации в этой версии не применяются.

Notebook читается по порядку: подготовка текста → словарный baseline v1 → сравнение с Random Forest и Logistic Regression на test → диагностика внутри train → ручной аудит → уточнённый baseline v2 на validation → сравнение старого и текущего способов обучения → итог. Версия v1 сохранена как точка сравнения. У v1 и ML есть test-результаты; v2 пока оценивается только на validation разработки.

Для запуска откройте notebook с рабочей папкой notebooks и выполните Restart Kernel & Run All. Нужны файлы в соседней папке data, библиотеки pandas, NumPy, Natasha, NLTK, scikit-learn и ресурс NLTK stopwords. Если ресурса нет, один раз выполните import nltk; nltk.download('stopwords'). Для воспроизведения согласованного аудита используется сохранённый train_annotation_audit.csv. Полный запуск проверен в Python 3.10.20; версии основных библиотек и ограничения результатов указаны ниже.


In [1]:
import re
import numpy as np
import pandas as pd
from IPython.display import display


In [2]:
user_query = pd.read_csv("../data/dataset_users_query.csv")
answer = pd.read_csv("../data/answers.csv")

In [3]:
answer.drop(["Unnamed: 0"], axis=1, inplace=True)

In [4]:
# Исходный query сохраняем для сопоставления с разметкой.
user_query['query_normalized'] = user_query['query'].str.lower()

In [5]:
# Одна и та же нормализация для запросов и словарных выражений.
def normalize_text(text):
    return re.sub(r'[\W_]+', ' ', text.lower()).strip()

user_query['query_normalized'] = user_query['query_normalized'].apply(normalize_text)


In [6]:
# Дубликаты считаем как число повторов после первого вхождения.
def summarize_text_columns(data, columns):
    summary = []
    for column in columns:
        values = data[column]
        non_missing = values.dropna()
        summary.append({
            "column": column,
            "rows": len(values),
            "missing": int(values.isna().sum()),
            "empty": int(non_missing.str.strip().eq("").sum()),
            "unique": int(non_missing.nunique()),
            "duplicates": int(non_missing.duplicated().sum()),
        })
    return pd.DataFrame(summary).set_index("column")

summarize_text_columns(user_query, ["query", "query_normalized"])


,rows,missing,empty,unique,duplicates
column,,,,,
query,17370,0,0,17370,0
query_normalized,17370,0,0,17370,0


In [7]:
from natasha import Doc, Segmenter, NewsEmbedding, NewsMorphTagger, MorphVocab

segmenter = Segmenter()
morph_tagger = NewsMorphTagger(NewsEmbedding())
morph_vocab = MorphVocab()

def lemmatize_text(text):
    if not text.strip():
        return ''
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    for token in doc.tokens:
        token.lemmatize(morph_vocab)
    return ' '.join([token.lemma for token in doc.tokens])

user_query['query_lemmatized'] = user_query['query_normalized'].apply(lemmatize_text)



C:\Users\POmpon\AppData\Local\Temp\codex-query-nlp-validation-01a11b1d\py310\lib\site-packages\pymorphy2\analyzer.py:114: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [8]:
from nltk.corpus import stopwords

# При первом запуске нужен ресурс NLTK: import nltk; nltk.download('stopwords').
nltk_stop_words = set(stopwords.words('russian'))
custom_stop_words = ['работа', 'вакансия', 'требуется', 'поиск', 'резюме', 'найти', 'предлагать', 'скачать']

# Приводим дополнительные стоп-слова к тем же леммам, что и запросы.
custom_stop_word_lemmas = {
    lemma
    for word in custom_stop_words
    for lemma in lemmatize_text(normalize_text(word)).split()
}
# Отрицания могут менять смысл условий работы. Не удаляем их.
protected_words = {'не', 'ни', 'нет', 'без'}
all_stop_words = (nltk_stop_words | custom_stop_word_lemmas) - protected_words

def remove_stop_words_nltk(text):
    return ' '.join(word for word in text.split() if word not in all_stop_words)

# Этот вариант сохраняем для будущих экспериментов с ML.
# Словарный baseline использует query_lemmatized с сохранёнными словами.
user_query['query_cleaned'] = user_query['query_lemmatized'].apply(remove_stop_words_nltk)
user_query


,query,query_normalized,query_lemmatized,query_cleaned
0,фарпост работа владивосток,фарпост работа владивосток,фарпост работа владивосток,фарпост владивосток
1,кофейни вакансии,кофейни вакансии,кофейня вакансия,кофейня
2,работа разнорабочие часовой,работа разнорабочие часовой,работа разнорабочий часовой,разнорабочий часовой
3,личный водитель на день,личный водитель на день,личный водитель на день,личный водитель день
4,работа от работодателя персональный водитель,работа от работодателя персональный водитель,работа от работодатель персональный водитель,работодатель персональный водитель
...,...,...,...,...
17365,япония вакансии,япония вакансии,япония вакансия,япония
17366,япония работа,япония работа,япония работа,япония
17367,японский язык работа,японский язык работа,японский язык работа,японский язык
17368,яппи вакансии,яппи вакансии,яппи вакансия,яппи


In [9]:
# Проверяем все этапы, сохраняя исходное число строк.
text_columns = ["query", "query_normalized", "query_lemmatized", "query_cleaned"]
preprocessing_summary = summarize_text_columns(user_query, text_columns)
preprocessing_summary


,rows,missing,empty,unique,duplicates
column,,,,,
query,17370,0,0,17370,0
query_normalized,17370,0,0,17370,0
query_lemmatized,17370,0,0,15937,1433
query_cleaned,17370,0,48,10720,6650


### Словарный baseline v1

Это назначение категорий по словарям, а не алгоритм кластеризации. Исходные списки терминов сохранены; сокращения внутри названий больше не теряются при разборе. Запросы и термины нормализуются и лемматизируются одинаково, стоп-слова для правил не удаляются.

Сопоставляем целые токены или последовательности токенов: `вахта` не совпадает с `вахтер`, а `ежедневная оплата` учитывается как выражение. Более длинное выражение проверяется раньше короткого, чтобы `системный администратор` не заменялся общим `администратор`. Для совместимости с исходным notebook пока сохраняем одно совпадение на категорию.

Ограничения: правила не понимают отрицания, синонимы и опечатки автоматически. Поэтому `без вахты` всё ещё может получить метку занятости «вахта». Сохранение отрицаний в preprocessing не решает эту задачу само по себе. Отсутствие совпадений пока наследует старую метку «общая фраза»; это признак отсутствия словарного совпадения, а не проверенная смысловая категория.

Таблица примеров ниже показывает поведение правил, включая ограничения. Она не является независимой оценкой качества. Для обучения далее используются метки из `answers.csv`; словарные предсказания сравниваются с ними на том же test, что и ML-модели.


In [10]:
import re

text_prof = " ААварийный комиссарАвтодиагностАвтомалярАвтоподборщикАвтор студенческих работАвторазборщикАвтослесарьАвтоэкспертАвтоэлектрикАгент по организации обслуживания пассажировАгент по организации почтово-грузовых авиаперевозокАгрономАдминистраторАккумуляторщикАктер, актрисаАкушерАналитикАниматорАппаратчикАрматурщикАрт-директорАртистАрхивариусАрхитекторАссистентАсфальтобетонщикАудиторББалетмейстерБанщикБаристаБарменБетонщикБиблиографБиблиотекарьБилетерБоецБоцманБригадирБурильщикБуфетчикБуфетчик судовойБухгалтерВВальцовщикВальщик леса, лесорубВарщик пищевого сырья и продуктовВахтерВедущий мероприятийВедущий специалист-экспертВесовщикВзрывникВидеографВидеомонтажерВизажистВодительВодолазВоеннослужащий по контрактуВожатыйВожатый служебных собакВокалистВоспитательВрачВыгульщик собакВышивальщица, вышивальщикВязальщикГГабионщикГазорезчикГальваникГардеробщикГеодезистГеологГигиенист стоматологическийГид, экскурсоводГидравлистГидромониторщикГладильщикГорнорабочийГорный мастерГрузчикГрумерГувернантка, гувернерДДворникДежурный по железнодорожной станцииДежурный по переездуДезинфекторДекларант, таможенный брокерДекораторДелопроизводительДетейлерДефектологДефектоскопистДизайнерДиректорДиспетчерДневальныйДозировщикДознавательДокер-механизаторДокмейстерДокументоведДомработница, домработникДонкерманДорожный мастерДорожный рабочийДоярка, доярДробильщикДрожжеводЖЖаровщикЖестянщикЖивотноводЖиловщик мясаЖурналистЗЗаведующийЗагрузчик-выгрузчикЗакальщик мороженогоЗакройщикЗамерщикЗаместительЗаправщик автокондиционеровЗаправщик АЗСЗаправщик картриджейЗаточникЗвукооператорЗвукорежиссерЗемлекопЗооняня, рабочий по уходу за животнымиЗоотехникЗуборезчикИИзготовитель конфетИзготовитель творогаИзолировщикИнженерИнкассаторИнспекторИнспектор службы безопасностиИнспектор транспортной безопасностиИнструкторИнтервьюерИнтернет-маркетологИнъектировщикИхтиопатологККабельщикКазначейКальянщикКаменщикКапитанКассирКастелянша, заведующий бельевойКинологКладовщикКлееварКолористКольщик дровКомендантКомплектовщикКомпрессорщикКондитерКондукторКонсультантКонсьержКонтролерКонюхКоординаторКопирайтерКоптильщикКоренщикКорреспондентКосильщикКосметик-эстетистКосметологКостюмерКочегарКредитный брокерКредитный специалистКредитный экспертКровельщикКрупьеКузнецКузовщикКурьерКухонный работникЛЛаборантЛепщик полуфабрикатовЛесникЛифтерЛогистММалярМаркетологМаркировщикМаркшейдерМассажистМастер аварийно-ремонтной службыМастер буровойМастер депиляцииМастер контрольныйМастер лесозаготовокМастер ногтевого сервисаМастер общестроительных работМастер отделочных работМастер откМастер по добычеМастер по изготовлению РВДМастер по ламинированию ресницМастер по наращиванию волосМастер по наращиванию ресницМастер по обработке камняМастер по обработке рыбыМастер по перетяжке салонов автомобилейМастер по ремонтуМастер по санитарному содержаниюМастер по удалению вмятин без покраскиМастер по эксплуатацииМастер погрузочно-разгрузочных работМастер производстваМастер производственного обученияМастер смрМастер татуировкиМастер участкаМастер цехаМастер-бровистМатросМашинистМедицинский представительМедрегистраторМедсестра, медбратМенеджерМерчендайзерМетеорологМетодистМетодологМетрологМеханизаторМеханикМикробиологМодельМодельерМодераторМойщикМонтажникМонтер путиМонтировщик сценыМотомеханикМотористМузыкальный работникМузыкальный руководительМузыкантМясникННабивщикНаладчикНаучный сотрудникНяняООбвальщикОбивщикОбмотчик элементов электрических машинОвощеводОгнеупорщикОзеленительОклейщикОперативный дежурныйОператорОперационистООперуполномоченныйОптикОрганизаторОсветитель, светотехникОсмотрщик вагоновОтделочникОфициантОформительОхранникОценщикППарикмахерПарковщикПастухПедагогПекарьПереводчикПескоструйщикПечатникПечникПивоварПильщикПлиточникПлотникПоварПодготовитель пищевого сырья и материаловПодносчик багажаПодручный сталевараПожарныйПолиграфологПолировщикПолицейскийПомощникПортнойПочтальонПрачечникПредставитель банкаПреподавательПрессовщикПриемосдатчик груза и багажаПриемщикПровизорПроводник пассажирского вагонаПрограммистПродавецПроектировщикПромоутерПромывальщик геологических пробПромышленный альпинистПрорабПроходчикПсихологПтицеводПтичница, птичникПчеловодРРаботник производственного цехаРаботник торгового залаРабочийРадиомеханикРадиомонтажник судовойРадиооператорРамщикРаскройщикРаспиловщикРаспространительРастениеводРевизорРегистраторРегулировщик радиоэлектронной аппаратурыРегулировщик скорости движения вагоновРедакторРежиссерРезчикРекламный агентРемонтник искусственных сооруженийРентгенолаборантРепетиторРеставраторРефмашинистРитуальный агентРубщик судовойРуководитель, начальникРулевойРыбообработчикССадовникСанитарСборщикСварщикСекретарьСервисный консультантСетевязальщикСигналистСиделкаСистемный администраторСистемотехникСледовательСлесарьСливщик-разливщикСметчикСмотрительСомелье, кавистСопровождающийСортировщикСоставитель поездовСоставитель смесейСоставитель фаршаСотрудник линии раздачиСпециалист абонентского отделаСпециалист аппарата мирового судьиСпециалист АХОСпециалист информационного отделаСпециалист клиентских подключенийСпециалист отдела кадровСпециалист отдела оперативной службыСпециалист отдела розничного бизнесаСпециалист отдела технического сопровожденияСпециалист паспортного столаСпециалист планово-экономического отделаСпециалист по взысканию задолженности / КоллекторСпециалист по воинскому учетуСпециалист по вооружениюСпециалист по госзакупкамСпециалист по гостеприимствуСпециалист по гражданской обороне и чрезвычайным ситуациямСпециалист по замене маслаСпециалист по защите информацииСпециалист по изготовлению мебелиСпециалист по изготовлению наружной рекламыСпециалист по изготовлению стеклопакетовСпециалист по информационной безопасностиСпециалист по коррекции фигурыСпециалист по мобилизационной работеСпециалист по обслуживаниюСпециалист по обслуживанию юридических лицСпециалист по организации мультимодальных перевозокСпециалист по охране трудаСпециалист по пластиковым картамСпециалист по предпродажной подготовкеСпециалист по работе с детьмиСпециалист по работе с населениемСпециалист по работе с семьейСпециалист по развал-схождениюСпециалист по ремонту автостеколСпециалист по ремонту стартеров и генераторовСпециалист по ремонту топливной аппаратурыСпециалист по ремонту трансмиссииСпециалист по ремонту ходовой частиСпециалист по сертификатамСпециалист по системам телефонииСпециалист по согласованиямСпециалист по социальной работеСпециалист по тендерамСпециалист по укладке асфальтаСпециалист по укладке тротуарной плитки, брусчаткиСпециалист по управлению многоквартирными домамиСпециалист по учебно-методической работеСпециалист по химчисткеСпециалист по экономической безопасностиСпециалист сервисного обслуживанияСпециалист службы безопасностиСпециалист строительного контроляСпециалист технической поддержкиСпециалист фрахтового отделаСпециалист-экспертСтаночникСтекольщикСтивидорСтилистСтолярСторожСтраховой агентСтрелокСтяжечникСу-шефСудебный приставСудовой агентСудокорпусник-ремонтникСупервайзерСуперинтендантСюрвейерТТабельщикТайный покупательТакелажникТальманТанцовщица, танцовщикТеплотехникТермистТестировщикТестоводТехникТехник зубнойТехник-лаборантТехник-энергетикТехнический специалистТехнический экспертТехнологТовароведТокарьТонировщикТорговый агентТорговый представительТранспортерщикТранспортировщикТрейдерТренерТрубопроводчик линейныйТрубопроводчик судовойТьюторУУборщик, горничнаяУкладчикУпаковщикУправляющийУстановщикУчастковый уполномоченный полицииУчетчикУчительФФактуровщикФармацевтФасовщикФельдшерФельдъегерьФизикФинансистФинансовый консультантФискаристФлористФормовщикФотографФрезеровщикФутеровщикХХимикХореографХостесХудожникЧЧертежникЧистильщикШШвейцарШвеяШелкографШиномонтажникШишкарьШлаковщикШлифовщикШтукатурЭЭкипировщикЭкологЭкономистЭкспедиторЭксперт по недвижимостиЭлектрикЭлектрогазосварщикЭлектромеханикЭлектромонтажникЭлектромонтерЭлектросварщикЭлектрослесарьЭнергетикЭтикетировщикЮЮвелирЮрисконсультЮрист"
text_busyness = "ДневнаяВечерняяНочнаяПосуточнаяПосменнаяПодработкаУдаленная ДомВахтаВременная ВыходнымНа неполный деньДень"
text_additional = "ИнвалидСтудентШкольникПенсионерМужчинаЖенщина"
text_conditions = "Ежедневная оплатаПроживание"

def text_split(text):
    # Названия начинаются с заглавной и строчной букв.
    # Сокращения АХО, АЗС и РВД остаются внутри названия.
    # Убираем буквенные заголовки: работникЛЛаборант -> работникЛаборант.
    text = re.sub(r'([а-яё])([А-ЯЁ])(?=\2[а-яё])', r'\1', text)
    entries = re.split(r'(?=[А-ЯЁ][а-яё])', text)
    return [
        term.strip()
        for entry in entries
        if len(entry.strip()) > 2
        for term in entry.split(',')
        if term.strip()
    ]

def prepare_terms(terms):
    normalized = [lemmatize_text(normalize_text(term)) for term in terms]
    unique_terms = list(dict.fromkeys(term for term in normalized if term.strip()))
    # Сначала более конкретные многословные выражения; при равной длине
    # сохраняем исходный порядок. По-прежнему возвращаем одну метку на категорию.
    return sorted(unique_terms, key=lambda term: -len(term.split()))

professions = text_split(text_prof)
employment_terms = text_split(text_busyness)
additional_terms = text_split(text_additional)
conditions_terms = text_split(text_conditions)

professions_cleaned = prepare_terms(professions)
employment_terms_cleaned = prepare_terms(employment_terms)
additional_terms_cleaned = prepare_terms(additional_terms)
conditions_terms_cleaned = prepare_terms(conditions_terms)


In [11]:
# Контролируем, сколько вариантов совпало после нормализации и лемматизации.
dictionary_summary = pd.DataFrame([
    {
        'category': category,
        'raw_terms': len(raw_terms),
        'prepared_terms': len(prepared_terms),
        'collapsed_variants': len(raw_terms) - len(prepared_terms),
    }
    for category, raw_terms, prepared_terms in [
        ('Employment', employment_terms, employment_terms_cleaned),
        ('Professions', professions, professions_cleaned),
        ('Additional', additional_terms, additional_terms_cleaned),
        ('Conditions', conditions_terms, conditions_terms_cleaned),
    ]
]).set_index('category')
dictionary_summary


,raw_terms,prepared_terms,collapsed_variants
category,,,
Employment,13,13,0
Professions,539,539,0
Additional,6,6,0
Conditions,2,2,0


In [12]:
def contains_token_sequence(query_tokens, term_tokens):
    if not term_tokens:
        return False
    length = len(term_tokens)
    return any(
        query_tokens[start:start + length] == term_tokens
        for start in range(len(query_tokens) - length + 1)
    )

# Название функции оставлено из исходной версии; она назначает словарные метки.
def assign_clusters(query, cluster_terms):
    query_tokens = query.split()
    matched_clusters = []
    for cluster, terms in cluster_terms.items():
        for term in terms:
            if contains_token_sequence(query_tokens, term.split()):
                matched_clusters.extend([cluster, term])
                break
    return matched_clusters if matched_clusters else [0]

cluster_terms = {
    'Employment': employment_terms_cleaned,
    'Professions': professions_cleaned,
    'Additional': additional_terms_cleaned,
    'Conditions': conditions_terms_cleaned,
}


In [13]:
import pandas as pd
import numpy as np

# Создание нового датафрейма с нужной структурой
result_df = pd.DataFrame(columns=['query', 'занятость', 'по должности-лемме', 'по дополнительному признаку', 'по условиям', 'общие фразы'])

# Функция для заполнения строк в новом датафрейме
def fill_clusters(row):
    # Правила читают леммы без удаления слов из многословных выражений.
    query = row['query_lemmatized']
    clusters = assign_clusters(query, cluster_terms)
    ##print(clusters)
    # По умолчанию все значения 0
    occupancy = position_lemma = additional = conditions = 0
    general_phrase = 0

    # Проверка, какие кластеры совпадают с запросом
    if 'Employment' in clusters:
        occupancy = clusters[clusters.index('Employment') + 1]  
    if 'Professions' in clusters:
        position_lemma = clusters[clusters.index('Professions') + 1]  
    if 'Additional' in clusters:
        additional = clusters[clusters.index('Additional') + 1] 
    if 'Conditions' in clusters:
        conditions = clusters[clusters.index('Conditions') + 1] 
    if 0 in clusters:
        general_phrase = 'общая фраза'  

    return pd.Series([query, occupancy, position_lemma, additional, conditions, general_phrase])

# Применение функции к каждому запросу и добавление результатов в новый датафрейм
result_df[['query', 'занятость', 'по должности-лемме', 'по дополнительному признаку', 'по условиям', 'общие фразы']] = user_query.apply(fill_clusters, axis=1)

# Вывод результата
result_df


,query,занятость,по должности-лемме,по дополнительному признаку,по условиям,общие фразы
0,фарпост работа владивосток,0,0,0,0,общая фраза
1,кофейня вакансия,0,0,0,0,общая фраза
2,работа разнорабочий часовой,0,0,0,0,общая фраза
3,личный водитель на день,день,водитель,0,0,0
4,работа от работодатель персональный водитель,0,водитель,0,0,0
...,...,...,...,...,...,...
17365,япония вакансия,0,0,0,0,общая фраза
17366,япония работа,0,0,0,0,общая фраза
17367,японский язык работа,0,0,0,0,общая фраза
17368,яппи вакансия,0,0,0,0,общая фраза


In [14]:
# Примеры для проверки совпадений и обсуждения ограничений baseline.
baseline_examples = pd.DataFrame({'query': [
    'вахтер',
    'вахта',
    'кухонным работником',
    'с ежедневной оплатой',
    'системным администратором',
    'специалист АХО',
    'без вахты',
]})
baseline_examples['query_normalized'] = baseline_examples['query'].apply(normalize_text)
baseline_examples['query_lemmatized'] = baseline_examples['query_normalized'].apply(lemmatize_text)
baseline_examples['rule_matches'] = baseline_examples['query_lemmatized'].apply(
    lambda query: assign_clusters(query, cluster_terms)
)
baseline_examples


,query,query_normalized,query_lemmatized,rule_matches
0,вахтер,вахтер,вахтер,"[Professions, вахтер]"
1,вахта,вахта,вахта,"[Employment, вахта]"
2,кухонным работником,кухонным работником,кухонный работник,"[Professions, кухонный работник]"
3,с ежедневной оплатой,с ежедневной оплатой,с ежедневный оплата,"[Conditions, ежедневный оплата]"
4,системным администратором,системным администратором,системный администратор,"[Professions, системный администратор]"
5,специалист АХО,специалист ахо,специалист ахо,"[Professions, специалист ахо]"
6,без вахты,без вахты,без вахта,"[Employment, вахта]"


In [15]:
# Показываем, что хранится в исходном файле разметки.
answer.head()


,query,занятость,по должности-лемме,по дополнительному признаку,по условиям,общие фразы
0,фарпост работа владивосток,NaN,NaN,NaN,NaN,общая фраза
1,кофейни вакансии,NaN,NaN,NaN,NaN,общая фраза
2,работа разнорабочие часовой,NaN,Рабочий,NaN,NaN,NaN
3,личный водитель на день,на неполный день,Водитель,NaN,NaN,NaN
4,работа от работодателя персональный водитель,NaN,Водитель,NaN,NaN,NaN


### Протокол оценки

Практическая задача — предсказать наличие пяти категорий в запросе. Метрики не оценивают правильность названия конкретной профессии или извлечение условий.

Метки берём из `answers.csv`. Пустую ячейку интерпретируем как отсутствие категории в файле, непустую — как её наличие. Результаты отражают совпадение предсказанных категорий с разметкой в файле.

Размеченные строки сопоставляем с исходными запросами по точному `query`. Запросы без строки в `answers.csv` не объявляем отрицательными примерами и не используем для расчёта метрик.

До векторизации выделяем train/test с фиксированным `random_state=42`. Группируем по `query_cleaned`: запросы, совпавшие после удаления стоп-слов, целиком попадают в одну часть. Это также исключает пересечение одинаковых лемматизированных запросов. Пустые `query_cleaned` образуют одну группу; строки сохраняются, поскольку модели используют непустой `query_lemmatized`.

GroupShuffleSplit выделяет 20% групп, поэтому доля строк в test может отличаться от 20%. Это одна фиксированная оценка без подбора параметров по test. Рядом показываем размеры частей, распределение меток и противоречия внутри групп. TF-IDF обучается только на train внутри Pipeline. Предобученная Natasha и фиксированные словари не подгоняются по test.

Сравниваем словарный baseline v1, Random Forest и Logistic Regression на одних и тех же запросах test. Для ML используем леммы с сохранёнными словами. Вариант без стоп-слов пока используется для группировки; его влияние на качество предстоит проверить отдельным экспериментом.


In [16]:
target_cols = ['занятость', 'по должности-лемме', 'по дополнительному признаку', 'по условиям', 'общие фразы']

def encode_label_presence(frame):
    # В answers.csv отсутствие категории записано как пропуск, в baseline — как 0.
    values = frame.fillna('').astype(str).apply(lambda column: column.str.strip())
    return (values.ne('') & values.ne('0')).astype('uint8')

assert answer['query'].notna().all() and answer['query'].is_unique
assert user_query['query'].notna().all() and user_query['query'].is_unique
assert set(target_cols).issubset(answer.columns)

feature_cols = ['query_normalized', 'query_lemmatized', 'query_cleaned']
evaluation_df = answer[['query'] + target_cols].merge(
    user_query[['query'] + feature_cols],
    on='query', how='left', validate='one_to_one', indicator=True,
)
assert evaluation_df['_merge'].eq('both').all(), 'Не все размеченные запросы найдены в датасете'
evaluation_df = evaluation_df.drop(columns='_merge')
assert evaluation_df['query_lemmatized'].str.strip().ne('').all()

# Цели для ML получаем только из файла разметки, а не из result_df.
y = encode_label_presence(evaluation_df[target_cols])
X = evaluation_df['query_lemmatized']
unlabeled_queries = user_query.loc[~user_query['query'].isin(answer['query'])].copy()

# Сопоставляем предсказания правил по исходному query, а не по леммам.
assert result_df.index.equals(user_query.index)
rule_labels = result_df[target_cols].copy()
rule_labels.insert(0, 'query', user_query['query'].to_numpy())
rule_y = encode_label_presence(
    evaluation_df[['query']].merge(rule_labels, on='query', validate='one_to_one')[target_cols]
)

dataset_summary = pd.Series({
    'all_queries': len(user_query),
    'labeled_queries': len(evaluation_df),
    'unlabeled_queries': len(unlabeled_queries),
    'labeled_without_categories': int(y.sum(axis=1).eq(0).sum()),
    'labeled_with_multiple_categories': int(y.sum(axis=1).gt(1).sum()),
}, name='rows')
dataset_summary


all_queries                         17370
labeled_queries                     14245
unlabeled_queries                    3125
labeled_without_categories              0
labeled_with_multiple_categories     1709
Name: rows, dtype: int64

In [17]:
# Противоречия сохраняем как диагностику; метки автоматически не исправляем.
groups = evaluation_df['query_cleaned']
group_sizes = groups.value_counts()
labels_per_group = y.groupby(groups).nunique()
conflicting_groups = labels_per_group.gt(1).any(axis=1)

group_summary = pd.Series({
    'groups': groups.nunique(),
    'groups_with_multiple_queries': int(group_sizes.gt(1).sum()),
    'groups_with_conflicting_labels': int(conflicting_groups.sum()),
    'queries_in_conflicting_groups': int(groups.isin(conflicting_groups[conflicting_groups].index).sum()),
    'empty_cleaned_queries': int(groups.eq('').sum()),
}, name='count')
group_summary


groups                            10477
groups_with_multiple_queries       2380
groups_with_conflicting_labels        3
queries_in_conflicting_groups        13
empty_cleaned_queries                13
Name: count, dtype: int64

In [18]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_indices, test_indices = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]

assert set(train_indices).isdisjoint(test_indices)
assert len(train_indices) + len(test_indices) == len(evaluation_df)
assert set(groups.iloc[train_indices]).isdisjoint(groups.iloc[test_indices])
assert set(X_train).isdisjoint(X_test)
assert y_train.nunique().eq(2).all(), 'В train должна быть представлена каждая категория и её отсутствие'

split_summary = pd.DataFrame({
    'rows': [len(train_indices), len(test_indices)],
    'groups': [groups.iloc[train_indices].nunique(), groups.iloc[test_indices].nunique()],
}, index=['train', 'test'])
split_summary['row_fraction'] = split_summary['rows'] / len(evaluation_df)
split_summary


,rows,groups,row_fraction
train,11362,8381,0.797613
test,2883,2096,0.202387


In [19]:
label_distribution = pd.DataFrame({
    'train_positive': y_train.sum(),
    'test_positive': y_test.sum(),
    'train_share': y_train.mean(),
    'test_share': y_test.mean(),
})
label_distribution


,train_positive,test_positive,train_share,test_share
занятость,1261,290,0.110984,0.100590
по должности-лемме,6253,1630,0.550343,0.565383
по дополнительному признаку,676,173,0.059497,0.060007
по условиям,474,92,0.041718,0.031911
общие фразы,4179,1044,0.367805,0.362123


In [20]:
from sklearn.metrics import accuracy_score, classification_report, f1_score, hamming_loss

def evaluate_predictions(y_true, y_pred):
    y_pred = np.asarray(y_pred)
    assert y_pred.shape == y_true.shape
    assert np.isin(y_pred, [0, 1]).all()
    scores = {
        'micro_f1': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'subset_accuracy': accuracy_score(y_true, y_pred),
        'hamming_loss': hamming_loss(y_true, y_pred),
    }
    report = classification_report(
        y_true, y_pred, target_names=target_cols, output_dict=True, zero_division=0,
    )
    per_label = pd.DataFrame(report).T.loc[target_cols, ['precision', 'recall', 'f1-score', 'support']]
    per_label.index.name = 'category'
    per_label['support'] = per_label['support'].astype(int)
    return scores, per_label

evaluation_results = {}
label_reports = {}
predictions = {}


In [21]:
# Правила не обучаются на answers.csv; оцениваем их на том же test.
predictions['Rule-based'] = rule_y.iloc[test_indices].to_numpy()
evaluation_results['Rule-based'], label_reports['Rule-based'] = evaluate_predictions(
    y_test, predictions['Rule-based'],
)
pd.Series(evaluation_results['Rule-based'], name='Rule-based')


micro_f1           0.891108
macro_f1           0.837556
weighted_f1        0.887585
subset_accuracy    0.859174
hamming_loss       0.047659
Name: Rule-based, dtype: float64

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline

rf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', MultiOutputClassifier(
        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    )),
])
rf_pipeline.fit(X_train, y_train)


,steps,"[('tfidf', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [23]:
predictions['Random Forest'] = rf_pipeline.predict(X_test)
evaluation_results['Random Forest'], label_reports['Random Forest'] = evaluate_predictions(
    y_test, predictions['Random Forest'],
)
pd.Series(evaluation_results['Random Forest'], name='Random Forest')


micro_f1           0.925229
macro_f1           0.961653
weighted_f1        0.925981
subset_accuracy    0.897329
hamming_loss       0.033368
Name: Random Forest, dtype: float64

In [24]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', MultiOutputClassifier(
        LogisticRegression(max_iter=1000, random_state=42),
    )),
])
lr_pipeline.fit(X_train, y_train)
predictions['Logistic Regression'] = lr_pipeline.predict(X_test)
evaluation_results['Logistic Regression'], label_reports['Logistic Regression'] = evaluate_predictions(
    y_test, predictions['Logistic Regression'],
)
pd.Series(evaluation_results['Logistic Regression'], name='Logistic Regression')


micro_f1           0.945381
macro_f1           0.959656
weighted_f1        0.945915
subset_accuracy    0.927159
hamming_loss       0.024280
Name: Logistic Regression, dtype: float64

In [25]:
# Это согласованность с имеющейся разметкой answers.csv, а не ручной проверкой.
comparison = pd.DataFrame.from_dict(evaluation_results, orient='index')
comparison.index.name = 'model'
comparison.round(4)


,micro_f1,macro_f1,weighted_f1,subset_accuracy,hamming_loss
model,,,,,
Rule-based,0.8911,0.8376,0.8876,0.8592,0.0477
Random Forest,0.9252,0.9617,0.9260,0.8973,0.0334
Logistic Regression,0.9454,0.9597,0.9459,0.9272,0.0243


In [26]:
per_label_comparison = pd.concat(label_reports, names=['model', 'category']).reset_index()
per_label_comparison.round(4)


,model,category,precision,recall,f1-score,support
0,Rule-based,занятость,0.9844,0.8690,0.9231,290
1,Rule-based,по должности-лемме,0.9824,0.8564,0.9151,1630
2,Rule-based,по дополнительному признаку,1.0000,0.4220,0.5935,173
3,Rule-based,по условиям,1.0000,0.7717,0.8712,92
4,Rule-based,общие фразы,0.8094,0.9761,0.8849,1044
5,Random Forest,занятость,1.0000,0.9897,0.9948,290
6,Random Forest,по должности-лемме,0.9800,0.8718,0.9227,1630
7,Random Forest,по дополнительному признаку,1.0000,0.9942,0.9971,173
8,Random Forest,по условиям,1.0000,1.0000,1.0000,92
9,Random Forest,общие фразы,0.8346,0.9617,0.8936,1044


In [27]:
# Новый запрос проходит ту же подготовку, что и тексты при обучении.
test_words = 'маляр работа студент'
test_text = lemmatize_text(normalize_text(test_words))
new_query_predictions = pd.DataFrame({
    'Random Forest': rf_pipeline.predict([test_text])[0],
    'Logistic Regression': lr_pipeline.predict([test_text])[0],
}, index=target_cols)
new_query_predictions


,Random Forest,Logistic Regression
занятость,0,0
по должности-лемме,1,1
по дополнительному признаку,1,0
по условиям,0,0
общие фразы,0,0


### Первоначальная оценка baseline v1 и ML-моделей

Запуск проверен 2026-10-09: Python 3.10.20, pandas 2.3.3, Natasha 1.6.0, NLTK 3.10.3, scikit-learn 1.7.2. Использовалось временное окружение Python 3.10 с setuptools 80.10.2 для совместимости pymorphy2; первоначальное окружение conda из старого notebook не использовалось.

После группового разделения в train — 11,362 запросов, в test — 2,883. Неразмеченные 3,125 запросов исключены из обучения и оценки. Пересечения групп и одинаковых лемм между частями нет. Проверка подтвердила, что 458 токенов, встречающихся только в test, отсутствуют в словарях TF-IDF обеих моделей; IDF соответствует статистике train.

По имеющейся разметке Logistic Regression показывает наибольший micro F1 (0.945), weighted F1 и subset accuracy. Random Forest немного выше по macro F1 (0.962 против 0.960). Поэтому вывод о победителе зависит от того, важнее ли общий результат или равный вес категорий.

У словарного baseline наиболее слабый recall в категории «по дополнительному признаку»: 0.422 на этом test. В исходном словаре нет выражения «без опыта», хотя оно встречается в answers.csv. Это гипотеза для дальнейшего разбора ошибок на train/validation; текущие словари после просмотра test не изменялись.

В 3 группах, содержащих 13 запросов, присутствуют разные наборы меток. Это может быть следствием как ошибок разметки, так и потери различий при очистке текста; автоматически исправлять такие случаи нельзя. Качество относительно независимой ручной разметки пока не установлено.


### Разбор ошибок внутри train

Для диагностики ML выделяем validation внутри ранее зафиксированного train. Группы query_cleaned сохраняются целиком, random_state=17. Клоны Pipeline обучаются только на внутреннем fit; основные модели, словари и финальный test не меняются.

Ошибки здесь означают расхождение с answers.csv, а не доказанную ошибку модели: возможны неточности разметки, потери при preprocessing и неполнота словарей. Ложный отрицательный ответ — категория есть в файле, но не предсказана; ложный положительный — предсказана, но отсутствует в файле. Оба типа требуют проверки исходного запроса.


In [28]:
from sklearn.base import clone

inner_splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=17)
fit_local, validation_local = next(inner_splitter.split(
    X_train, y_train, groups=groups.iloc[train_indices],
))
fit_indices = train_indices[fit_local]
validation_indices = train_indices[validation_local]
assert set(fit_indices).isdisjoint(validation_indices)
assert set(fit_indices) | set(validation_indices) == set(train_indices)
assert set(groups.iloc[fit_indices]).isdisjoint(groups.iloc[validation_indices])
assert set(validation_indices).isdisjoint(test_indices)

X_fit, X_validation = X.iloc[fit_indices], X.iloc[validation_indices]
y_fit, y_validation = y.iloc[fit_indices], y.iloc[validation_indices]
assert y_fit.nunique().eq(2).all()
validation_predictions = {'Rule-based': rule_y.iloc[validation_indices].to_numpy()}
diagnostic_models = {
    'Random Forest': clone(rf_pipeline),
    'Logistic Regression': clone(lr_pipeline),
}
for name, pipeline in diagnostic_models.items():
    pipeline.fit(X_fit, y_fit)
    validation_predictions[name] = pipeline.predict(X_validation)

inner_split_summary = pd.DataFrame({
    'rows': [len(fit_indices), len(validation_indices)],
    'groups': [groups.iloc[fit_indices].nunique(), groups.iloc[validation_indices].nunique()],
}, index=['fit', 'validation'])
inner_split_summary


,rows,groups
fit,9116,6704
validation,2246,1677


In [29]:
validation_scores = {}
error_summary_rows = []
validation_error_frames = []
truth = y_validation.to_numpy()
for model_name, predicted in validation_predictions.items():
    validation_scores[model_name], _ = evaluate_predictions(y_validation, predicted)
    missed = (truth == 1) & (predicted == 0)
    extra = (truth == 0) & (predicted == 1)
    for position, category in enumerate(target_cols):
        error_summary_rows.append({
            'model': model_name,
            'category': category,
            'positive_in_file': int(truth[:, position].sum()),
            'missed': int(missed[:, position].sum()),
            'extra': int(extra[:, position].sum()),
        })
    cases = evaluation_df.iloc[validation_indices][['query', 'query_lemmatized']].copy()
    cases['evaluation_index'] = validation_indices
    cases['model'] = model_name
    cases['missed_categories'] = [
        ', '.join(category for category, flag in zip(target_cols, row) if flag)
        for row in missed
    ]
    cases['extra_categories'] = [
        ', '.join(category for category, flag in zip(target_cols, row) if flag)
        for row in extra
    ]
    cases['label_errors'] = (missed | extra).sum(axis=1)
    validation_error_frames.append(cases.loc[cases['label_errors'].gt(0)])

validation_comparison = pd.DataFrame.from_dict(validation_scores, orient='index')
validation_comparison.index.name = 'model'
validation_error_summary = pd.DataFrame(error_summary_rows)
validation_error_examples = pd.concat(validation_error_frames, ignore_index=True)
validation_error_examples = validation_error_examples.sort_values(
    ['model', 'label_errors', 'query'], ascending=[True, False, True], kind='stable',
)
display(validation_comparison.round(4), validation_error_summary)


,micro_f1,macro_f1,weighted_f1,subset_accuracy,hamming_loss
model,,,,,
Rule-based,0.8907,0.8383,0.8881,0.8540,0.0480
Random Forest,0.9082,0.9503,0.9091,0.8780,0.0411
Logistic Regression,0.9337,0.9455,0.9343,0.9061,0.0294


,model,category,positive_in_file,missed,extra
0,Rule-based,занятость,255,31,13
1,Rule-based,по должности-лемме,1254,189,23
2,Rule-based,по дополнительному признаку,114,62,0
3,Rule-based,по условиям,88,22,0
4,Rule-based,общие фразы,817,28,171
5,Random Forest,занятость,255,3,1
6,Random Forest,по должности-лемме,1254,200,27
7,Random Forest,по дополнительному признаку,114,1,0
8,Random Forest,по условиям,88,0,2
9,Random Forest,общие фразы,817,43,184


In [30]:
# Только validation внутри train. По этим примерам нельзя оценивать финальный test.
for model_name in validation_predictions:
    print(model_name)
    display(validation_error_examples.loc[
        validation_error_examples['model'].eq(model_name),
        ['query', 'query_lemmatized', 'missed_categories', 'extra_categories'],
    ].head(4))


Rule-based


,query,query_lemmatized,missed_categories,extra_categories
183,работа в вайлдберриз удаленно без опыта,работа в вайлдберриз удаленно без опыт,"занятость, по дополнительному признаку",общие фразы
279,работа в сфере логистики без опыта,работа в сфера логистика без опыт,"по должности-лемме, по дополнительному признаку",общие фразы
50,работа сисадмином удаленно,работа сисадмин удаленно,"занятость, по должности-лемме",общие фразы
55,разнорабочий в ночь оплата сразу,разнорабочий в ночь оплата сразу,"по должности-лемме, по условиям",общие фразы


Random Forest


,query,query_lemmatized,missed_categories,extra_categories
438,1c разработчик вакансии,1 c разработчик вакансия,по должности-лемме,общие фразы
477,авиадиспетчер вакансии,авиадиспетчер вакансия,по должности-лемме,общие фразы
478,архитектор вакансия,архитектор вакансия,по должности-лемме,общие фразы
479,архитектор работа,архитектор работа,по должности-лемме,общие фразы


Logistic Regression


,query,query_lemmatized,missed_categories,extra_categories
704,авиадиспетчер вакансии,авиадиспетчер вакансия,по должности-лемме,общие фразы
705,архитектор вакансия,архитектор вакансия,по должности-лемме,общие фразы
706,архитектор работа,архитектор работа,по должности-лемме,общие фразы
707,банщик работа,банщик работа,по должности-лемме,общие фразы


### Наблюдения по разбору ошибок

Внутренний fit содержит 9 116 запросов, validation — 2 246. По имеющейся разметке micro F1 на validation равен 0.891 для baseline, 0.908 для Random Forest и 0.934 для Logistic Regression. Эти диагностические результаты не заменяют ранее зафиксированную оценку test; основные модели и её метрики не изменились.

Baseline пропускает 62 из 114 положительных примеров категории «по дополнительному признаку». Среди примеров расхождений есть «без опыта», сокращение «сисадмин» и наречие «удаленно». Одинаковая лемматизация не делает сокращения и синонимы одинаковыми: «сисадмин» не становится выражением «системный администратор», а «удаленно» не превращается в «удаленный». Это гипотезы для отдельных изменений словаря на train/validation, без настройки по test.

У ML основные расхождения относятся к профессиям и общим фразам. Например, для «архитектор вакансия» файл содержит профессиональную категорию, а обе диагностические модели предсказывают общую фразу. Нужен разбор представления текста и исходной разметки; само расхождение ещё не доказывает правильность файла.

Форма аудита содержит 50 случайных и 30 целевых запросов. Среди целевых есть 9 запросов со словами отрицания и 6 запросов из групп с противоречивыми метками; признаки отбора могут пересекаться. В train нет пустых query_cleaned для отбора: такие строки остались в целиком выделенной группе test, поэтому в форму они не включены. На момент подготовки проверено вручную 0 из 80 строк. Перезапуск сохраняет существующий файл, выбранные запросы и введённые человеком значения.


### Ручная проверка разметки train

Подготовлены 80 запросов: 50 случайных строк train и 30 целевых примеров с отрицаниями, противоречивыми метками, пустым query_cleaned или расхождениями правил/моделей с файлом. Случайная часть помогает исследовать обычные запросы; целевая ищет проблемные случаи. Их нельзя объединять для оценки общей доли ошибок разметки. Эта выборка находится в train и служит аудиту и уточнению правил, а не независимым тестовым набором.

В data/train_annotation_audit.csv отсутствуют исходные метки и предсказания. Сначала заполните review-поля по смыслу исходного query: 1 — категория присутствует, 0 — отсутствует, ? — неясно. Сохраните audit_id, query и sample_type. Статусы: pending — ещё не проверено; reviewed — все пять меток определены как 0/1; uncertain — требуется обсуждение. В review_comment запишите причину сомнения.

Критерии для этой проверки:

- Занятость: запрошенный формат или график работы. По согласованному правилу «без вахты» само по себе не даёт метку занятости; «без вахты, на неполный день» даёт её за второй формат. Работа в интернете относится к формату онлайн-занятости; это не обязательно работа из дома. «Работа до 5 вечера» задаёт ограничение графика и получает занятость = 1.
- По должности: явное название профессии или конкретного занятия, включая сокращения. По согласованным правилам SEO, Java/Python в контексте работы и вакансий, дизайн как конкретное занятие и описание работы «ремонт и установка кондиционеров» достаточны для присутствия категории профессии. Указание объекта работы («на корабле», «на котельной») без конкретного занятия эту метку не даёт. Широкая область «менеджмент» без конкретного занятия не даёт этой метки. Информационный запрос «работа риэлтором плюсы и минусы» получает по должности = 1: размечаем содержание, а не только намерение искать вакансию. Если профессия только подразумевается в других формулировках, отметьте ? и поясните неоднозначность.
- Дополнительный признак: опыт или характеристики кандидата. «Без опыта» — положительный запрос на возможность работать без опыта, поэтому эта категория присутствует. По согласованным правилам квалификационный разряд и категория водительских прав не относятся к этой категории: при отсутствии других дополнительных признаков ставим 0.
- Условия: явно запрошенные условия, например ежедневная оплата, явно указанная сумма зарплаты, проживание или официальное оформление. По согласованному правилу «официальная работа для девушки» получает условия = 1 и дополнительный признак = 1. Географию (города, страны) в эту категорию не включаем. По уточнению пользователя Васильков — название города, а не профессии. Указание объекта работы («на корабле», «на котельной») само по себе не означает проживание, вахту или другие условия. Отрицаемые условия сами по себе не считаем запрошенными; спорные формулировки отмечаем ?.
- Общая фраза: общий запрос о работе без определённых признаков остальных четырёх категорий. При неясной границе используем ?, не превращаем сомнение автоматически в 0.

audit_context хранит исходные метки и причины отбора отдельно от формы. Сравнение с файлом появляется только для строк со статусом reviewed и пятью заполненными бинарными метками. Исходный answers.csv автоматически не исправляется. При повторном запуске сохраняются уже выбранные строки и введённые вручную значения.

Аудит выполнен с помощью ассистента: предложения меток обсуждены и подтверждены пользователем. Это согласованная разметка train для проверки критериев, а не независимая слепая экспертная оценка или новая разметка всего датасета. Предложения были сделаны без просмотра исходных меток; запросы для целевой части отбирались с учётом расхождений с ними.


In [31]:
from pathlib import Path

audit_path = Path('../data/train_annotation_audit.csv')
audit_candidates = evaluation_df.iloc[train_indices].copy()
audit_candidates['evaluation_index'] = train_indices
audit_candidates['negation'] = audit_candidates['query_normalized'].str.contains(
    r'\b(?:не|ни|нет|без)\b', regex=True,
)
train_conflicts = y_train.groupby(audit_candidates['query_cleaned']).nunique().gt(1).any(axis=1)
audit_candidates['conflicting_labels'] = audit_candidates['query_cleaned'].isin(
    train_conflicts.loc[train_conflicts].index,
)
audit_candidates['empty_after_stopwords'] = audit_candidates['query_cleaned'].eq('')
audit_candidates['rule_disagreement'] = y_train.ne(rule_y.iloc[train_indices]).any(axis=1)
audit_candidates['validation_model_error'] = audit_candidates.index.isin(
    validation_error_examples.loc[
        validation_error_examples['model'].ne('Rule-based'), 'evaluation_index',
    ].unique(),
)
audit_reasons = ['negation', 'conflicting_labels', 'empty_after_stopwords',
                 'rule_disagreement', 'validation_model_error']

if audit_path.exists():
    # Зафиксированная выборка важнее нового отбора после изменения модели.
    existing_identity = pd.read_csv(audit_path, encoding='utf-8-sig', keep_default_na=False)
    assert {'audit_id', 'query', 'sample_type'}.issubset(existing_identity.columns)
    assert existing_identity['query'].is_unique and existing_identity['audit_id'].is_unique
    assert set(existing_identity['query']).issubset(audit_candidates['query'])
    audit_sample = existing_identity[['audit_id', 'query', 'sample_type']].merge(
        audit_candidates, on='query', validate='one_to_one',
    )
else:
    random_indices = audit_candidates.sample(n=50, random_state=20261009).index.tolist()
    selected = set(random_indices)
    targeted_indices = []
    for number, reason in enumerate(audit_reasons):
        pool = audit_candidates.loc[audit_candidates[reason] & ~audit_candidates.index.isin(selected)]
        picked = pool.sample(n=min(6, len(pool)), random_state=100 + number).index.tolist()
        targeted_indices.extend(picked)
        selected.update(picked)
    remaining = 30 - len(targeted_indices)
    pool = audit_candidates.loc[audit_candidates['rule_disagreement'] & ~audit_candidates.index.isin(selected)]
    picked = pool.sample(n=remaining, random_state=200).index.tolist()
    targeted_indices.extend(picked)
    audit_sample = audit_candidates.loc[random_indices + targeted_indices].copy()
    audit_sample['audit_id'] = audit_sample['evaluation_index'].map(lambda index: f'train-{index:05d}')
    audit_sample['sample_type'] = ['random'] * 50 + ['targeted'] * 30

assert len(audit_sample) == 80 and audit_sample['query'].is_unique
assert audit_sample['sample_type'].value_counts().to_dict() == {'random': 50, 'targeted': 30}
assert set(audit_sample['evaluation_index']).issubset(train_indices)
assert set(audit_sample['evaluation_index']).isdisjoint(test_indices)
review_cols = ['review_' + category for category in target_cols]
audit_draft = audit_sample[['audit_id', 'query', 'sample_type']].sort_values('audit_id').reset_index(drop=True)
for column in review_cols:
    audit_draft[column] = ''
audit_draft['review_status'] = 'pending'
audit_draft['review_comment'] = ''
audit_sample.groupby('sample_type').size().rename('queries')


sample_type
random      50
targeted    30
Name: queries, dtype: int64

In [32]:
# Исходные метки и признаки отбора отделены от формы ручной проверки.
audit_context = audit_sample[['audit_id', 'query', 'sample_type'] + feature_cols + audit_reasons].copy()
for category in target_cols:
    source_values = evaluation_df.set_index('query')[category]
    audit_context['annotation_' + category] = audit_context['query'].map(source_values).fillna('')

# Причины могут пересекаться, поэтому их суммы не равны числу целевых примеров.
audit_reason_summary = audit_sample.groupby('sample_type')[audit_reasons].sum()
audit_reason_summary


,negation,conflicting_labels,empty_after_stopwords,rule_disagreement,validation_model_error
sample_type,,,,,
random,0,0,0,6,3
targeted,9,6,0,25,12


In [33]:
def load_or_create_audit_form(path, draft):
    if not path.exists():
        draft.to_csv(path, index=False, encoding='utf-8-sig')
    # Существующий файл никогда не перезаписываем.
    form = pd.read_csv(path, encoding='utf-8-sig', keep_default_na=False, dtype=str)
    assert set(draft.columns).issubset(form.columns)
    assert form['audit_id'].is_unique and form['query'].is_unique
    expected = draft[['audit_id', 'query', 'sample_type']].set_index('audit_id').sort_index()
    actual = form[['audit_id', 'query', 'sample_type']].set_index('audit_id').sort_index()
    pd.testing.assert_frame_equal(actual, expected)
    assert form[review_cols].isin(['', '0', '1', '?']).all().all()
    assert form['review_status'].isin(['pending', 'reviewed', 'uncertain']).all()
    return form

audit_form = load_or_create_audit_form(audit_path, audit_draft)
completed_review = audit_form['review_status'].eq('reviewed') & audit_form[review_cols].isin(['0', '1']).all(axis=1)
uncertain_review = ~completed_review & (
    audit_form['review_status'].eq('uncertain') | audit_form[review_cols].eq('?').any(axis=1)
)
audit_progress = pd.Series({
    'selected': len(audit_form),
    'reviewed': int(completed_review.sum()),
    'uncertain': int(uncertain_review.sum()),
    'pending': int((~completed_review & ~uncertain_review).sum()),
    'incomplete_reviewed_status': int((audit_form['review_status'].eq('reviewed') & ~completed_review).sum()),
}, name='queries')
display(audit_progress)

if completed_review.any():
    reviewed_context = audit_form.loc[completed_review].merge(
        audit_context, on=['audit_id', 'query', 'sample_type'], validate='one_to_one',
    )
    audit_agreement_rows = []
    audit_query_rows = []
    audit_example_rows = []
    for sample_type, checked in reviewed_context.groupby('sample_type'):
        checked = checked.reset_index(drop=True)
        human_values = checked[review_cols].astype('uint8').to_numpy()
        source_values = encode_label_presence(
            checked[['annotation_' + category for category in target_cols]]
        ).to_numpy()
        differences = human_values != source_values
        audit_query_rows.append({
            'sample_type': sample_type,
            'reviewed_queries': len(checked),
            'matching_queries': int((~differences.any(axis=1)).sum()),
            'differing_queries': int(differences.any(axis=1).sum()),
            'differing_labels': int(differences.sum()),
        })
        for position, category in enumerate(target_cols):
            audit_agreement_rows.append({
                'sample_type': sample_type, 'category': category,
                'reviewed_queries': len(checked),
                'disagreements': int(differences[:, position].sum()),
                'review_only': int(((human_values[:, position] == 1) &
                                    (source_values[:, position] == 0)).sum()),
                'source_only': int(((human_values[:, position] == 0) &
                                    (source_values[:, position] == 1)).sum()),
            })
        for position in np.flatnonzero(differences.any(axis=1)):
            audit_example_rows.append({
                'audit_id': checked.iloc[position]['audit_id'],
                'query': checked.iloc[position]['query'],
                'sample_type': sample_type,
                'review_only_categories': ', '.join(
                    category for number, category in enumerate(target_cols)
                    if human_values[position, number] == 1 and source_values[position, number] == 0
                ),
                'source_only_categories': ', '.join(
                    category for number, category in enumerate(target_cols)
                    if human_values[position, number] == 0 and source_values[position, number] == 1
                ),
            })
    audit_query_summary = pd.DataFrame(audit_query_rows)
    audit_category_summary = pd.DataFrame(audit_agreement_rows)
    audit_disagreement_examples = pd.DataFrame(audit_example_rows, columns=[
        'audit_id', 'query', 'sample_type', 'review_only_categories', 'source_only_categories',
    ])
    display(audit_query_summary)
    display(audit_category_summary)
    # Случайные и целевые примеры показываем отдельно, без общей оценки доли ошибок.
    for sample_type, examples in audit_disagreement_examples.groupby('sample_type'):
        print(f'{sample_type}: примеры расхождений с существующей разметкой')
        display(examples.head(10))
else:
    print('Ручная проверка ещё не выполнена: метки не заполнены автоматически.')


selected                      80
reviewed                      80
uncertain                      0
pending                        0
incomplete_reviewed_status     0
Name: queries, dtype: int64

,sample_type,reviewed_queries,matching_queries,differing_queries,differing_labels
0,random,50,40,10,21
1,targeted,30,20,10,14


,sample_type,category,reviewed_queries,disagreements,review_only,source_only
0,random,занятость,50,2,2,0
1,random,по должности-лемме,50,4,4,0
2,random,по дополнительному признаку,50,4,4,0
3,random,по условиям,50,2,2,0
4,random,общие фразы,50,9,0,9
5,targeted,занятость,30,1,1,0
6,targeted,по должности-лемме,30,9,7,2
7,targeted,по дополнительному признаку,30,0,0,0
8,targeted,по условиям,30,0,0,0
9,targeted,общие фразы,30,4,1,3


random: примеры расхождений с существующей разметкой


,audit_id,query,sample_type,review_only_categories,source_only_categories
0,train-05195,официальная работа для девушки,random,"по дополнительному признаку, по условиям",общие фразы
1,train-05941,работа риэлтором плюсы и минусы,random,по должности-лемме,общие фразы
2,train-08616,вакансии java,random,по должности-лемме,общие фразы
3,train-11176,охрана судов вакансии,random,по должности-лемме,общие фразы
4,train-11354,подработка для подростков 16 лет москва,random,по дополнительному признаку,
5,train-11650,работа 100 000 рублей в месяц,random,по условиям,общие фразы
6,train-11888,работа в интернете для подростков вакансии,random,"занятость, по дополнительному признаку",общие фразы
7,train-12412,работа для подростков вакансии,random,по дополнительному признаку,общие фразы
8,train-12461,работа до 5 вечера,random,занятость,общие фразы
9,train-12524,работа коком,random,по должности-лемме,общие фразы


targeted: примеры расхождений с существующей разметкой


,audit_id,query,sample_type,review_only_categories,source_only_categories
10,train-01092,seo работа без опыта,targeted,по должности-лемме,
11,train-02097,работа мастером по шугарингу без опыта,targeted,по должности-лемме,
12,train-08693,вакансии в букмекерские конторы,targeted,общие фразы,по должности-лемме
13,train-08856,вакансии грумер,targeted,по должности-лемме,общие фразы
14,train-08936,вакансии ихтиопатолог,targeted,по должности-лемме,общие фразы
15,train-11683,работа python без опыта,targeted,по должности-лемме,
16,train-12189,работа вахта камчатка,targeted,,по должности-лемме
17,train-12498,работа из дома без опыта,targeted,занятость,
18,train-12573,работа мастер цеха в москве,targeted,по должности-лемме,общие фразы
19,train-13287,рекрутер вакансии удаленно,targeted,по должности-лемме,


### Результаты согласованного аудита train

Все 80 запросов обсуждены и подтверждены пользователем. Васильков в двух запросах — название города (по уточнению пользователя), поэтому это слово не добавляет профессию или условия работы.

| Часть выборки | Проверено запросов | Все пять меток совпали | Есть расхождения | Различающихся бинарных меток |
| --- | ---: | ---: | ---: | ---: |
| Случайная | 50 | 40 | 10 | 21 |
| Целевая | 30 | 20 | 10 | 14 |

Число различающихся меток может превышать число запросов: один запрос имеет несколько категорий. Эти две части не объединяем в оценку доли ошибок всего датасета. Случайная часть содержит лишь 50 запросов train; целевая специально обогащена проблемными случаями. Метки предложены ассистентом и подтверждены пользователем, поэтому аудит не является независимой слепой экспертной оценкой.

Различаем два типа расхождений:

- Явные несоответствия текста и исходной метки: в answers.csv у «работа вахта камчатка» указана профессия «Вахтер», а у «вакансии в букмекерские конторы» — «Кондуктор». Эти профессии в запросах отсутствуют. Эти примеры показывают несоответствия исходной разметки тексту запросов.
- Различия согласованных критериев и существующей разметки: «вакансии java» получает профессию в нашем аудите, «официальная работа для девушки» — дополнительный признак и условия, явно указанная зарплата — условия. В исходном файле эти запросы отнесены к общим фразам. Расхождения критериев рассматриваем отдельно от явных несоответствий текста и метки.

Существующие test-метрики по-прежнему измеряют совпадение с категориями в answers.csv. Аудит не заменяет test и не оправдывает исправление всего файла по 80 примерам. Исходная разметка, разделение train/test и сохранённые результаты моделей не изменены.

В следующем блоке baseline v2 приведён к согласованным критериям: варианты профессий и занятий, форматы работы, опыт и возраст, условия оплаты и отрицания. Проверки отдельных формулировок и сравнение на внутренней validation показаны отдельно от первоначальной оценки test. Аудит train служит для разработки правил; улучшение на этих 80 запросах само по себе не доказывает обобщение.


### Baseline версии 2: согласованные правила

Версия 1 выше остаётся исходной точкой сравнения; её сохранённые test-результаты относятся только к ней. Версия 2 добавляет варианты профессий и занятий, форматы онлайн-работы, опыт и возраст, условия оплаты. Словари версии 1 не меняются.

Сопоставляем целые последовательности слов в нормализованном исходном тексте и в леммах Natasha. Это сохраняет границы слов и помогает при контекстной ошибке лемматизации. Общие слова «дом» и «день» заменены явными выражениями формата работы: отдельное упоминание дома или дня не задаёт занятость.

Для занятости и условий проверяем локальное отрицание перед совпадением и некоторые конструкции после него. Запятые, разделители и «а/но» отделяют части запроса. «Без опыта» — положительный дополнительный признак; профессия и характеристики кандидата размечаются по упоминанию. Это ограниченные правила: сложное отрицание, кавычки, двойное отрицание и дальние связи могут потребовать отдельного разбора. «Общая фраза» в предсказании остаётся результатом отсутствия совпадений остальных категорий и может скрывать пропущенный признак.

Правила разработаны по согласованному аудиту train. Улучшение на этих 80 примерах — проверка реализации, а не доказательство обобщения. Для сравнения на внутренней validation исключаем группы query_cleaned, представленные в аудите. Validation служит разработке правил и использует существующую разметку answers.csv; это не независимый экспертный тест. Отложенный test, ML-модели и исходная разметка на этом этапе не меняются.


In [34]:
from functools import lru_cache

@lru_cache(maxsize=20000)
def rule_token_variants(text):
    normalized = normalize_text(text)
    variants = (tuple(normalized.split()), tuple(lemmatize_text(normalized).split()))
    return tuple(dict.fromkeys(tokens for tokens in variants if tokens))

def prepare_rule_patterns(terms):
    patterns = dict.fromkeys(tokens for term in terms for tokens in rule_token_variants(term))
    return sorted(patterns, key=lambda tokens: -len(tokens))

profession_aliases_v2 = [
    'сисадмин', 'seo', 'java', 'python', 'дизайн', 'риэлтор', 'риелтор',
    'рекрутер', 'охотовед', 'кок', 'уборщица', 'уборка', 'сестра хозяйка',
    'мастер по шугарингу', 'специалист по контекстной рекламе',
    'специалист по наращиванию ресниц', 'ремонт кондиционеров',
    'установка кондиционеров', 'сборка ручек', 'охрана судов',
]
employment_aliases_v2 = [
    'удаленно', 'удалённо', 'дистанционно', 'на дому', 'из дома',
    'работа дома', 'работать дома', 'в интернете', 'онлайн',
    'неполный день', 'неполный рабочий день', 'по выходным',
    'утром', 'вечером', 'ночью', 'вахтой', 'вахту', 'вахты',
]
additional_aliases_v2 = [
    'без опыта', 'с опытом', 'опыт работы', 'подросток', 'девушка', 'парень',
]
conditions_aliases_v2 = [
    'оплата ежедневно', 'каждодневная оплата', 'оплата каждый день',
    'официальная работа', 'официальное оформление', 'официальное трудоустройство',
]
rule_patterns_v2 = {
    'занятость': prepare_rule_patterns([
        term for term in employment_terms if normalize_text(term) not in {'дом', 'день'}
    ] + employment_aliases_v2),
    'по должности-лемме': prepare_rule_patterns(professions + profession_aliases_v2),
    'по дополнительному признаку': prepare_rule_patterns(additional_terms + additional_aliases_v2),
    'по условиям': prepare_rule_patterns(conditions_terms + conditions_aliases_v2),
}

negation_prefix_v2 = re.compile(
    r'(?:^| )(?:не|ни|нет|без)(?: (?:хотеть|хочу|хотим|искать|ищу|'
    r'рассматривать|нужный|нужен|нужна|нужны|работа|работать|на|в|с|за)){0,5}$'
)
negation_postfixes_v2 = {
    ('не', 'нужный'), ('не', 'нужен'), ('не', 'нужна'), ('не', 'нужны'),
    ('не', 'требоваться'), ('не', 'требуется'),
}

def rule_match_is_negated(tokens, start, length):
    prefix = ' '.join(tokens[max(0, start - 6):start])
    suffix = tuple(tokens[start + length:start + length + 2])
    return bool(negation_prefix_v2.search(prefix)) or suffix in negation_postfixes_v2

def find_rule_match(token_variants, patterns, check_negation=False):
    for pattern in patterns:
        length = len(pattern)
        for tokens in token_variants:
            for start in range(len(tokens) - length + 1):
                if tuple(tokens[start:start + length]) != pattern:
                    continue
                if check_negation and rule_match_is_negated(tokens, start, length):
                    continue
                return ' '.join(pattern)
    return 0

age_pattern_v2 = re.compile(r'\b(\d{1,3})\s*(?:лет|год|года)\b')
time_pattern_v2 = re.compile(
    r'\b(?:до|с)\s*(?:[01]?\d|2[0-3])\s*(?:утра|дня|вечера|ночи|часа|часов)\b'
)
salary_pattern_v2 = re.compile(
    r'(?<!\w)\d+(?:[ \u00a0]\d{3})*(?:[.,]\d+)?\s*'
    r'(?:тыс(?:яч[аиу])?\.?\s*)?(?:руб(?:ль|ля|лей)?\.?|₽)(?!\w)'
)

def classify_query_rules_v2(query):
    # Разделители сохраняют область отрицания до нормализации текста.
    clauses = re.split(
        r'[;!?]|(?<!\d),(?!\d)|(?<!\d)(?<!тыс)(?<!руб)\.(?!\d)|\b(?:но|а)\b',
        query.lower(),
    )
    tags = dict.fromkeys(target_cols, 0)
    for clause in clauses:
        tokens = rule_token_variants(clause)
        for category, patterns in rule_patterns_v2.items():
            if not tags[category]:
                tags[category] = find_rule_match(
                    tokens, patterns, check_negation=category in {'занятость', 'по условиям'},
                )
        for category, pattern in [
            ('занятость', time_pattern_v2),
            ('по дополнительному признаку', age_pattern_v2),
            ('по условиям', salary_pattern_v2),
        ]:
            if tags[category]:
                continue
            for match in pattern.finditer(clause):
                if category == 'по дополнительному признаку' and not 14 <= int(match.group(1)) <= 100:
                    continue
                value = find_rule_match(
                    tokens, prepare_rule_patterns([match.group(0)]),
                    check_negation=category in {'занятость', 'по условиям'},
                )
                if value:
                    tags[category] = value
                    break
    if not any(tags[category] for category in target_cols[:-1]):
        tags['общие фразы'] = 'общая фраза'
    return tags

def predict_rule_tags_v2(queries):
    return pd.DataFrame(
        [classify_query_rules_v2(query) for query in queries],
        columns=target_cols, index=queries.index,
    )


In [35]:
# Проверки смысла правил, границ слов и области отрицания.
rule_checks_v2 = [
    ('без вахты', [0, 0, 0, 0, 1]),
    ('вахтер без опыта', [0, 1, 1, 0, 0]),
    ('вахта охранник', [1, 1, 0, 0, 0]),
    ('без вахты, на неполный день', [1, 0, 0, 0, 0]),
    ('работа без опыта вахтой', [1, 0, 1, 0, 0]),
    ('не хочу работать вахтой', [0, 0, 0, 0, 1]),
    ('вахта не нужна', [0, 0, 0, 0, 1]),
    ('вахта не нужна, подработка нужна', [1, 0, 0, 0, 0]),
    ('не вахта, а подработка', [1, 0, 0, 0, 0]),
    ('неполный день', [1, 0, 0, 0, 0]),
    ('без проживания', [0, 0, 0, 0, 1]),
    ('не нужна ежедневная оплата', [0, 0, 0, 0, 1]),
    ('с проживанием без ежедневной оплаты', [0, 0, 0, 1, 0]),
    ('без проживания, оплата ежедневно', [0, 0, 0, 1, 0]),
    ('работа с ежедневной оплатой', [0, 0, 0, 1, 0]),
    ('работа продавец в детском доме', [0, 1, 0, 0, 0]),
    ('слесарь 3 разряда', [0, 1, 0, 0, 0]),
    ('водитель категории се', [0, 1, 0, 0, 0]),
    ('работа 100 000 рублей в месяц', [0, 0, 0, 1, 0]),
    ('работа 120 тыс. руб. в месяц', [0, 0, 0, 1, 0]),
    ('работа до 5 вечера', [1, 0, 0, 0, 0]),
    ('работа до 17 часов', [1, 0, 0, 0, 0]),
    ('работа с 16 лет', [0, 0, 1, 0, 0]),
    ('работа васильков без опыта', [0, 0, 1, 0, 0]),
    ('вакансии java', [0, 1, 0, 0, 0]),
    ('работа python без опыта', [0, 1, 1, 0, 0]),
    ('работа электромеханик', [0, 1, 0, 0, 0]),
    ('вакансии в букмекерские конторы', [0, 0, 0, 0, 1]),
    ('работа в интернете для подростков', [1, 0, 1, 0, 0]),
]
checked_queries_v2 = pd.Series([query for query, expected in rule_checks_v2])
checked_predictions_v2 = encode_label_presence(predict_rule_tags_v2(checked_queries_v2))
expected_checks_v2 = np.array([expected for query, expected in rule_checks_v2], dtype='uint8')
failed_checks_v2 = np.flatnonzero((checked_predictions_v2.to_numpy() != expected_checks_v2).any(axis=1))
assert len(failed_checks_v2) == 0, [
    (rule_checks_v2[position][0], rule_checks_v2[position][1],
     checked_predictions_v2.iloc[position].tolist(),
     classify_query_rules_v2(rule_checks_v2[position][0]))
    for position in failed_checks_v2
]
print(f'Пройдено проверок baseline v2: {len(rule_checks_v2)}')
display(pd.concat([
    checked_queries_v2.rename('query'), checked_predictions_v2,
], axis=1).head(10))


Пройдено проверок baseline v2: 29


,query,занятость,по должности-лемме,по дополнительному признаку,по условиям,общие фразы
0,без вахты,0,0,0,0,1
1,вахтер без опыта,0,1,1,0,0
2,вахта охранник,1,1,0,0,0
3,"без вахты, на неполный день",1,0,0,0,0
4,работа без опыта вахтой,1,0,1,0,0
5,не хочу работать вахтой,0,0,0,0,1
6,вахта не нужна,0,0,0,0,1
7,"вахта не нужна, подработка нужна",1,0,0,0,0
8,"не вахта, а подработка",1,0,0,0,0
9,неполный день,1,0,0,0,0


In [36]:
# Внутренняя validation используется для разработки, без групп из ручного аудита.
assert completed_review.all(), 'Сначала завершите согласование формы аудита'
audited_groups_v2 = set(
    evaluation_df.loc[evaluation_df['query'].isin(audit_form['query']), 'query_cleaned']
)
baseline_dev_indices = validation_indices[
    ~groups.iloc[validation_indices].isin(audited_groups_v2).to_numpy()
]
assert set(baseline_dev_indices).isdisjoint(test_indices)
assert set(groups.iloc[baseline_dev_indices]).isdisjoint(audited_groups_v2)
baseline_dev_summary = pd.Series({
    'validation_before_exclusion': len(validation_indices),
    'excluded_rows_from_audited_groups': len(validation_indices) - len(baseline_dev_indices),
    'development_validation_rows': len(baseline_dev_indices),
}, name='queries')
display(baseline_dev_summary)

dev_cases_v2 = evaluation_df.iloc[baseline_dev_indices]
v1_dev_tags = dev_cases_v2.apply(fill_clusters, axis=1)
v1_dev_tags.columns = ['query'] + target_cols
v1_dev_binary = encode_label_presence(v1_dev_tags[target_cols])
v2_dev_binary = encode_label_presence(predict_rule_tags_v2(dev_cases_v2['query']))
dev_truth_v2 = y.iloc[baseline_dev_indices]
assert v1_dev_binary.index.equals(dev_truth_v2.index)
assert v2_dev_binary.index.equals(dev_truth_v2.index)
baseline_dev_scores = {}
baseline_dev_label_reports = {}
for name, predicted in [('Baseline v1', v1_dev_binary), ('Baseline v2', v2_dev_binary)]:
    baseline_dev_scores[name], baseline_dev_label_reports[name] = evaluate_predictions(dev_truth_v2, predicted)
baseline_dev_comparison = pd.DataFrame.from_dict(baseline_dev_scores, orient='index')
display(baseline_dev_comparison.round(4))
display(pd.concat(baseline_dev_label_reports, names=['version', 'category']))

# Аудит — примеры разработки правил; случайную и целевую части показываем отдельно.
audit_cases_v2 = audit_form.merge(
    evaluation_df[['query', 'query_lemmatized', 'query_cleaned']], on='query', validate='one_to_one',
)
assert set(audit_cases_v2['query']).issubset(evaluation_df.iloc[train_indices]['query'])
v1_audit_tags = audit_cases_v2.apply(fill_clusters, axis=1)
v1_audit_tags.columns = ['query'] + target_cols
v1_audit_binary = encode_label_presence(v1_audit_tags[target_cols])
v2_audit_binary = encode_label_presence(predict_rule_tags_v2(audit_cases_v2['query']))
audit_truth_v2 = audit_cases_v2[review_cols].astype('uint8').set_axis(target_cols, axis=1)
baseline_audit_rows = []
for sample_type in ['random', 'targeted']:
    mask = audit_cases_v2['sample_type'].eq(sample_type)
    for name, predicted in [('Baseline v1', v1_audit_binary), ('Baseline v2', v2_audit_binary)]:
        scores, _ = evaluate_predictions(audit_truth_v2.loc[mask], predicted.loc[mask])
        baseline_audit_rows.append({
            'sample_type': sample_type, 'version': name, 'queries': int(mask.sum()),
            'matching_queries': int(predicted.loc[mask].eq(audit_truth_v2.loc[mask]).all(axis=1).sum()),
            'micro_f1': scores['micro_f1'],
        })
baseline_audit_comparison = pd.DataFrame(baseline_audit_rows)
display(baseline_audit_comparison)


validation_before_exclusion          2246
excluded_rows_from_audited_groups      43
development_validation_rows          2203
Name: queries, dtype: int64

,micro_f1,macro_f1,weighted_f1,subset_accuracy,hamming_loss
Baseline v1,0.8990,0.8483,0.8961,0.8625,0.0444
Baseline v2,0.9157,0.9055,0.9163,0.8897,0.0378


precision    recall  f1-score  \
version     category                                                     
Baseline v1 занятость                     0.944444  0.884000  0.913223   
            по должности-лемме            0.980447  0.859592  0.916050   
            по дополнительному признаку   1.000000  0.456140  0.626506   
            по условиям                   1.000000  0.795181  0.885906   
            общие фразы                   0.840300  0.967901  0.899598   
Baseline v2 занятость                     0.902256  0.960000  0.930233   
            по должности-лемме            0.980090  0.884082  0.929614   
            по дополнительному признаку   0.786207  1.000000  0.880309   
            по условиям                   0.985294  0.807229  0.887417   
            общие фразы                   0.868966  0.933333  0.900000   

                                         support  
version     category                              
Baseline v1 занятость                        250  
            по должности-лемме              1225  
            по дополнительному признаку      114  
            по условиям                       83  
            общие фразы                      810  
Baseline v2 занятость                        250  
            по должности-лемме              1225  
            по дополнительному признаку      114  
            по условиям                       83  
            общие фразы                      810

,sample_type,version,queries,matching_queries,micro_f1
0,random,Baseline v1,50,34,0.707965
1,random,Baseline v2,50,50,1.000000
2,targeted,Baseline v1,30,10,0.500000
3,targeted,Baseline v2,30,30,1.000000


### Что изменилось после уточнения baseline

Сравнение выполнено на 2 203 запросах внутренней validation с метками из answers.csv. Из исходных 2 246 запросов исключены 43 строки: все группы query_cleaned, пересекающиеся с ручным аудитом. Обе версии оценены на одном и том же наборе.

| Метрика | Baseline v1 | Baseline v2 |
|---|---:|---:|
| Micro F1 | 0.8990 | 0.9157 |
| Macro F1 | 0.8483 | 0.9055 |
| Weighted F1 | 0.8961 | 0.9163 |
| Все пять меток совпали (subset accuracy) | 0.8625 | 0.8897 |
| Доля ошибочных бинарных меток (hamming loss) | 0.0444 | 0.0378 |

Наибольший прирост F1 — у дополнительных признаков: 0,627 → 0,880. Полнота относительно исходной разметки выросла с 0,456 до 1,000, а precision снизился с 1,000 до 0,786. Для занятости полнота выросла с 0,884 до 0,960, precision снизился с 0,944 до 0,902. Более широкие правила находят пропущенные признаки, но чаще расходятся с answers.csv; без дополнительной проверки нельзя разделить ошибки правил, ошибки исходной разметки и различия критериев.

Все 29 проверок границ слов, отрицаний и согласованных признаков прошли. На 80 запросах ручного аудита полное совпадение изменилось с 44/80 до 80/80: в случайной части — с 34/50 до 50/50, в целевой — с 10/30 до 30/30. Эти примеры использованы при разработке правил, поэтому результат 80/80 подтверждает реализацию согласованных решений и не является оценкой качества на новых запросах.

Validation также используется для разработки. Полученные F1 отражают совпадение с категориями в answers.csv. Не оценивается точность извлечения конкретного названия профессии. Для следующей оценки качества по согласованным критериям нужна отдельная разметка, не использованная при настройке правил. Сохранённые test-метрики выше относятся к baseline v1 и прежним ML-моделям; baseline v2 на test в этом шаге не оценивался.


### Как читать результаты

Micro F1 суммирует решения по категориям, macro F1 даёт каждой категории одинаковый вес, weighted F1 учитывает число положительных примеров. Для редких категорий дополнительно смотрим precision, recall и support. Subset accuracy требует полного совпадения всех пяти меток запроса; Hamming loss показывает долю ошибочных решений по отдельным меткам и должен быть ниже.

Полученные числа нельзя напрямую сравнивать с F1 исходного notebook: изменились источник целевых меток, правила, группировка и схема векторизации. По одному split нельзя судить об устойчивости результатов. Противоречивые метки внутри групп требуют отдельного разбора.

Test используется для итогового сравнения текущих подходов. Подбор словарей, порогов и гиперпараметров должен выполняться на train/validation; для дальнейшей независимой проверки нужна отдельно проверенная разметка, которая не будет участвовать в настройке.

Неразмеченные запросы сохранены в unlabeled_queries. У них нет эталонных меток для расчёта F1. Проверка названий профессий, обработка отрицаний, оценка влияния стоп-слов и ручной аудит answers.csv остаются отдельными задачами.


## Сравнение способов обучения до и после исправлений

Проверяем гипотезу: текущие ML-модели лучше согласуются с метками answers.csv, чем модели, обученные старым способом. Это контролируемая реконструкция логики из коммита ce9ce57, а не буквальный повтор прежних F1: у всех вариантов одинаковые строки обучения, модели, параметры, версии библиотек и критерий оценки.

Обучаем на внутреннем fit из 9 116 запросов. Оцениваем на тех же 2 203 запросах validation, что и baseline v2; группы ручного аудита исключены из оценки. Отложенный test не участвует в эксперименте. Старый проект обучался на большем наборе, включая запросы без answers.csv; здесь число обучающих строк фиксировано, чтобы сравнить способы, а не объём данных.

| Вариант | Представление текста | Обучающие метки | Обучение TF-IDF |
|---|---|---|---|
| A0 legacy leaky | Исходная очистка и стоп-слова | Исходные словарные псевдометки | fit + validation |
| A1 legacy strict | Исходная очистка и стоп-слова | Исходные словарные псевдометки | Только fit |
| B source labels | Исходная очистка и стоп-слова | answers.csv | Только fit |
| C current | Текущие леммы с сохранёнными словами | answers.csv | Только fit |

A0 специально воспроизводит утечку для диагностики и не является корректным протоколом обучения. A1 → B показывает влияние замены псевдометок на имеющуюся разметку; B → C — влияние обработки текста. Основное корректное сравнение способов — A1 → C. Baseline v2 не создаёт метки для ML и не участвует в их обучении.

Все варианты проверяются по одному answers.csv, включая модели, обученные на другом источнике меток. Так мы проверяем качество относительно выбранного эталона. Результат относится к меткам выбранной validation и задаче определения наличия категорий.


In [37]:
# Восстановление исходной обработки текста и генерации псевдометок.
def legacy_text_split(text):
    entries = re.findall(r'[А-Я][^А-Я]*', text)
    filtered = [entry for entry in entries if len(entry) > 2]
    return [part.strip() for entry in filtered for part in entry.split(',')]

legacy_terms = [
    [term.lower() for term in legacy_text_split(text)]
    for text in [text_busyness, text_prof, text_additional, text_conditions]
]
legacy_stop_words = set(stopwords.words('russian')) | {
    'работа', 'вакансия', 'требуется', 'поиск', 'резюме', 'найти',
    'работник', 'предлагать', 'скачать',
}

def prepare_legacy_text(query):
    normalized = re.sub(r'[^а-яА-ЯёЁa-zA-Z0-9\s]', '', query.lower()).strip()
    lemmas = lemmatize_text(normalized)
    return ' '.join(word for word in lemmas.split() if word not in legacy_stop_words)

def legacy_pseudo_labels(text):
    flags = [int(any(term in text for term in terms)) for terms in legacy_terms]
    return flags + [int(not any(flags))]

ab_fit_indices = fit_indices.copy()
ab_validation_indices = baseline_dev_indices.copy()
assert set(ab_fit_indices).isdisjoint(ab_validation_indices)
assert set(groups.iloc[ab_fit_indices]).isdisjoint(groups.iloc[ab_validation_indices])
assert set(groups.iloc[ab_validation_indices]).isdisjoint(audited_groups_v2)
assert set(ab_fit_indices).union(ab_validation_indices).isdisjoint(test_indices)
assert len(ab_fit_indices) == 9116 and len(ab_validation_indices) == 2203

# Текст готовится только для fit и validation; test здесь не обрабатывается.
ab_all_indices = np.concatenate([ab_fit_indices, ab_validation_indices])
ab_raw_queries = evaluation_df.iloc[ab_all_indices]['query']
ab_legacy_texts = ab_raw_queries.apply(prepare_legacy_text)
ab_legacy_labels = pd.DataFrame(
    ab_legacy_texts.apply(legacy_pseudo_labels).tolist(),
    index=ab_legacy_texts.index, columns=target_cols, dtype='uint8',
)
ab_fit_legacy = ab_legacy_texts.loc[X.iloc[ab_fit_indices].index]
ab_val_legacy = ab_legacy_texts.loc[X.iloc[ab_validation_indices].index]
ab_fit_pseudo = ab_legacy_labels.loc[ab_fit_legacy.index]
ab_fit_truth = y.iloc[ab_fit_indices]
ab_validation_truth = y.iloc[ab_validation_indices]
assert ab_fit_pseudo.nunique().eq(2).all()
assert ab_fit_pseudo.index.equals(ab_fit_truth.index)
assert ab_val_legacy.index.equals(ab_validation_truth.index)
assert not ab_fit_pseudo.equals(ab_fit_truth)
ab_data_summary = pd.Series({
    'fit_rows': len(ab_fit_indices),
    'validation_rows': len(ab_validation_indices),
    'validation_groups': groups.iloc[ab_validation_indices].nunique(),
    'fit_rows_with_different_pseudo_and_source_labels': int(
        ab_fit_pseudo.ne(ab_fit_truth).any(axis=1).sum()
    ),
    'empty_legacy_fit_texts': int(ab_fit_legacy.eq('').sum()),
    'empty_legacy_validation_texts': int(ab_val_legacy.eq('').sum()),
}, name='count')
display(ab_data_summary)


fit_rows                                            9116
validation_rows                                     2203
validation_groups                                   1656
fit_rows_with_different_pseudo_and_source_labels    1458
empty_legacy_fit_texts                                 1
empty_legacy_validation_texts                          0
Name: count, dtype: int64

In [38]:
# Параметры RF/LR одинаковы во всех вариантах; основные модели не заменяем.
ab_predictions = {}
ab_scores = {}
ab_label_reports = {}
ab_vectorizer_checks = []
ab_model_templates = {'Random Forest': rf_pipeline, 'Logistic Regression': lr_pipeline}
ab_variants = {
    'A0 legacy leaky': (ab_fit_legacy, ab_val_legacy, ab_fit_pseudo, True),
    'A1 legacy strict': (ab_fit_legacy, ab_val_legacy, ab_fit_pseudo, False),
    'B source labels': (ab_fit_legacy, ab_val_legacy, ab_fit_truth, False),
    'C current': (X.iloc[ab_fit_indices], X.iloc[ab_validation_indices], ab_fit_truth, False),
}
for variant, (fit_texts, validation_texts, fit_labels, leaky) in ab_variants.items():
    for model_name, template in ab_model_templates.items():
        pipeline = clone(template)
        if leaky:
            vectorizer = pipeline.named_steps['tfidf']
            vectorizer.fit(pd.concat([fit_texts, validation_texts]))
            classifier = pipeline.named_steps['classifier']
            classifier.fit(vectorizer.transform(fit_texts), fit_labels)
            predicted = classifier.predict(vectorizer.transform(validation_texts))
        else:
            pipeline.fit(fit_texts, fit_labels)
            predicted = pipeline.predict(validation_texts)
            vectorizer = pipeline.named_steps['tfidf']
        analyzer = vectorizer.build_analyzer()
        fit_tokens = {t for text in fit_texts for t in analyzer(text)}
        val_tokens = {t for text in validation_texts for t in analyzer(text)}
        validation_only_tokens = val_tokens - fit_tokens
        leaked = set(vectorizer.vocabulary_) & validation_only_tokens
        if not leaky:
            assert not leaked
            assert set(vectorizer.vocabulary_) == fit_tokens
        else:
            assert set(vectorizer.vocabulary_) == fit_tokens | val_tokens
        key = (model_name, variant)
        ab_predictions[key] = predicted
        ab_scores[key], ab_label_reports[key] = evaluate_predictions(ab_validation_truth, predicted)
        ab_vectorizer_checks.append({
            'model': model_name, 'variant': variant,
            'validation_only_tokens_in_vocabulary': len(leaked),
        })
        print(f'{model_name}: {variant}, micro F1 = {ab_scores[key]["micro_f1"]:.4f}')

ab_comparison = pd.DataFrame.from_dict(ab_scores, orient='index')
ab_comparison.index = pd.MultiIndex.from_tuples(ab_comparison.index, names=['model', 'variant'])
ab_per_label = pd.concat(ab_label_reports, names=['model', 'variant', 'category'])
display(ab_comparison.round(4))
display(pd.DataFrame(ab_vectorizer_checks))
display(ab_per_label.round(4))

# Контроль: текущий вариант повторяет ранее рассчитанные прогнозы клона на validation.
ab_validation_positions = np.flatnonzero(np.isin(validation_indices, ab_validation_indices))
assert np.array_equal(validation_indices[ab_validation_positions], ab_validation_indices)
for name in ab_model_templates:
    assert np.array_equal(
        ab_predictions[(name, 'C current')],
        validation_predictions[name][ab_validation_positions],
    )


Random Forest: A0 legacy leaky, micro F1 = 0.8663


Logistic Regression: A0 legacy leaky, micro F1 = 0.8596


Random Forest: A1 legacy strict, micro F1 = 0.8680


Logistic Regression: A1 legacy strict, micro F1 = 0.8613


Random Forest: B source labels, micro F1 = 0.9460


...

Random Forest: C current, micro F1 = 0.9141


Logistic Regression: C current, micro F1 = 0.9399


,,micro_f1,macro_f1,weighted_f1,subset_accuracy,hamming_loss
model,variant,,,,,
Random Forest,A0 legacy leaky,0.8663,0.7548,0.8609,0.8198,0.0586
Logistic Regression,A0 legacy leaky,0.8596,0.7327,0.8523,0.8148,0.0611
Random Forest,A1 legacy strict,0.8680,0.7561,0.8626,0.8225,0.0578
Logistic Regression,A1 legacy strict,0.8613,0.7343,0.8539,0.8166,0.0604
Random Forest,B source labels,0.9460,0.9661,0.9464,0.9260,0.0243
Logistic Regression,B source labels,0.9407,0.9493,0.9410,0.9187,0.0265
Random Forest,C current,0.9141,0.9530,0.9148,0.8847,0.0385
Logistic Regression,C current,0.9399,0.9496,0.9403,0.9133,0.0267


,model,variant,validation_only_tokens_in_vocabulary
0,Random Forest,A0 legacy leaky,401
1,Logistic Regression,A0 legacy leaky,401
2,Random Forest,A1 legacy strict,0
3,Logistic Regression,A1 legacy strict,0
4,Random Forest,B source labels,0
5,Logistic Regression,B source labels,0
6,Random Forest,C current,0
7,Logistic Regression,C current,0


precision  \
model               variant          category                                 
Random Forest       A0 legacy leaky  занятость                       0.8935   
                                     по должности-лемме              0.9865   
                                     по дополнительному признаку     1.0000   
                                     по условиям                     1.0000   
                                     общие фразы                     0.7833   
Logistic Regression A0 legacy leaky  занятость                       0.9317   
                                     по должности-лемме              0.9892   
                                     по дополнительному признаку     1.0000   
                                     по условиям                     1.0000   
                                     общие фразы                     0.7780   
Random Forest       A1 legacy strict занятость                       0.9019   
                                     по должности-лемме              0.9884   
                                     по дополнительному признаку     1.0000   
                                     по условиям                     1.0000   
                                     общие фразы                     0.7861   
Logistic Regression A1 legacy strict занятость                       0.9324   
                                     по должности-лемме              0.9892   
                                     по дополнительному признаку     1.0000   
                                     по условиям                     1.0000   
                                     общие фразы                     0.7811   
Random Forest       B source labels  занятость                       0.9650   
                                     по должности-лемме              0.9903   
                                     по дополнительному признаку     1.0000   
                                     по условиям                     0.9651   
                                     общие фразы                     0.8778   
Logistic Regression B source labels  занятость                       0.9717   
                                     по должности-лемме              0.9841   
                                     по дополнительному признаку     1.0000   
                                     по условиям                     1.0000   
                                     общие фразы                     0.8872   
Random Forest       C current        занятость                       0.9960   
                                     по должности-лемме              0.9748   
                                     по дополнительному признаку     1.0000   
                                     по условиям                     0.9765   
                                     общие фразы                     0.8212   
Logistic Regression C current        занятость                       0.9875   
                                     по должности-лемме              0.9910   
                                     по дополнительному признаку     0.9811   
                                     по условиям                     1.0000   
                                     общие фразы                     0.8904   

                                                                  recall  \
model               variant          category                              
Random Forest       A0 legacy leaky  занятость                    0.7720   
                                     по должности-лемме           0.8343   
                                     по дополнительному признаку  0.4561   
                                     по условиям                  0.3735   
                                     общие фразы                  0.9815   
Logistic Regression A0 legacy leaky  занятость                    0.7640   
                                     по должности-лемме           0.8212   
                                     по дополнительному признак

In [39]:
# Парный bootstrap по группам: одни и те же группы для всех предсказаний.
# Интервал отражает вариативность оценки на этой validation, а не переобучение моделей.
ab_group_codes, ab_unique_groups = pd.factorize(groups.iloc[ab_validation_indices], sort=True)
ab_keys = list(ab_predictions)
ab_group_counts = []
ab_true_values = ab_validation_truth.to_numpy()
for key in ab_keys:
    predicted = ab_predictions[key]
    per_query = np.column_stack([
        ((ab_true_values == 1) & (predicted == 1)).sum(axis=1),
        ((ab_true_values == 0) & (predicted == 1)).sum(axis=1),
        ((ab_true_values == 1) & (predicted == 0)).sum(axis=1),
    ])
    grouped = np.zeros((len(ab_unique_groups), 3), dtype='int64')
    np.add.at(grouped, ab_group_codes, per_query)
    ab_group_counts.append(grouped)
ab_group_counts = np.stack(ab_group_counts, axis=1)
ab_bootstrap_rng = np.random.default_rng(20261010)
ab_bootstrap_f1 = []
ab_bootstrap_repetitions = 2000
for offset in range(0, ab_bootstrap_repetitions, 100):
    size = min(100, ab_bootstrap_repetitions - offset)
    selected = ab_bootstrap_rng.integers(
        0, len(ab_unique_groups), size=(size, len(ab_unique_groups)),
    )
    totals = ab_group_counts[selected].sum(axis=1)
    tp, fp, fn = totals[:, :, 0], totals[:, :, 1], totals[:, :, 2]
    ab_bootstrap_f1.append(2 * tp / (2 * tp + fp + fn))
ab_bootstrap_f1 = np.concatenate(ab_bootstrap_f1)
assert ab_bootstrap_f1.shape == (ab_bootstrap_repetitions, len(ab_keys))
ab_contrasts = [
    ('A0 legacy leaky', 'C current'),
    ('A1 legacy strict', 'C current'),
    ('A0 legacy leaky', 'A1 legacy strict'),
    ('A1 legacy strict', 'B source labels'),
    ('B source labels', 'C current'),
]
ab_delta_rows = []
for model_name in ab_model_templates:
    for earlier, later in ab_contrasts:
        first, second = ab_keys.index((model_name, earlier)), ab_keys.index((model_name, later))
        differences = ab_bootstrap_f1[:, second] - ab_bootstrap_f1[:, first]
        low, high = np.quantile(differences, [0.025, 0.975])
        ab_delta_rows.append({
            'model': model_name, 'from': earlier, 'to': later,
            'micro_f1_delta': ab_scores[(model_name, later)]['micro_f1']
                              - ab_scores[(model_name, earlier)]['micro_f1'],
            'ci95_low': low, 'ci95_high': high,
        })
ab_deltas = pd.DataFrame(ab_delta_rows)
display(ab_deltas.round(4))


,model,from,to,micro_f1_delta,ci95_low,ci95_high
0,Random Forest,A0 legacy leaky,C current,0.0478,0.0323,0.0646
1,Random Forest,A1 legacy strict,C current,0.0461,0.0303,0.0628
2,Random Forest,A0 legacy leaky,A1 legacy strict,0.0017,-0.0001,0.0037
3,Random Forest,A1 legacy strict,B source labels,0.0780,0.0635,0.0927
4,Random Forest,B source labels,C current,-0.0319,-0.0422,-0.0227
5,Logistic Regression,A0 legacy leaky,C current,0.0803,0.0678,0.0945
6,Logistic Regression,A1 legacy strict,C current,0.0786,0.0658,0.0925
7,Logistic Regression,A0 legacy leaky,A1 legacy strict,0.0017,0.0004,0.0032
8,Logistic Regression,A1 legacy strict,B source labels,0.0794,0.0667,0.0937
9,Logistic Regression,B source labels,C current,-0.0008,-0.0060,0.0044


In [40]:
# Расхождения с файлом: улучшения и ухудшения показываем вместе.
ab_changed_examples = []
for model_name in ab_model_templates:
    old_prediction = ab_predictions[(model_name, 'A1 legacy strict')]
    current_prediction = ab_predictions[(model_name, 'C current')]
    old_matches = (old_prediction == ab_true_values).all(axis=1)
    current_matches = (current_prediction == ab_true_values).all(axis=1)
    for direction, mask in [
        ('текущий совпал, старый расходится', current_matches & ~old_matches),
        ('старый совпал, текущий расходится', old_matches & ~current_matches),
    ]:
        for position in np.flatnonzero(mask)[:3]:
            def categories(values):
                return ', '.join(category for category, flag in zip(target_cols, values) if flag)
            ab_changed_examples.append({
                'model': model_name, 'direction': direction,
                'query': evaluation_df.iloc[ab_validation_indices[position]]['query'],
                'source_categories': categories(ab_true_values[position]),
                'legacy_categories': categories(old_prediction[position]),
                'current_categories': categories(current_prediction[position]),
            })
ab_examples = pd.DataFrame(ab_changed_examples)
display(ab_examples)


,model,direction,query,source_categories,legacy_categories,current_categories
0,Random Forest,"текущий совпал, старый расходится",999 работа уборщица,по должности-лемме,общие фразы,по должности-лемме
1,Random Forest,"текущий совпал, старый расходится",работа по специальной оценке условий труда,по должности-лемме,общие фразы,по должности-лемме
2,Random Forest,"текущий совпал, старый расходится",работа в ночную смену для девушек,занятость,общие фразы,занятость
3,Random Forest,"старый совпал, текущий расходится",фарпост работа владивосток,общие фразы,общие фразы,
4,Random Forest,"старый совпал, текущий расходится",работа владивосток фарпост,общие фразы,общие фразы,
5,Random Forest,"старый совпал, текущий расходится",работа на складе свежие,общие фразы,общие фразы,
6,Logistic Regression,"текущий совпал, старый расходится",999 работа уборщица,по должности-лемме,общие фразы,по должности-лемме
7,Logistic Regression,"текущий совпал, старый расходится",работа по специальной оценке условий труда,по должности-лемме,общие фразы,по должности-лемме
8,Logistic Regression,"текущий совпал, старый расходится",работа в ночную смену для девушек,занятость,общие фразы,занятость
9,Logistic Regression,"старый совпал, текущий расходится",работа студентам 17 лет,по дополнительному признаку,по дополнительному признаку,


### Выводы сравнительного эксперимента

Одинаковый внутренний fit: 9 116 запросов; одинаковая validation: 2 203 запроса. Метки оценки во всех вариантах — answers.csv. Ниже показан micro F1; macro F1, полное совпадение категорий и показатели каждой категории приведены в таблицах выше.

| Вариант | Random Forest | Logistic Regression |
|---|---:|---:|
| Старый способ с утечкой TF-IDF | 0.8663 | 0.8596 |
| Старые тексты и псевдометки, без утечки | 0.8680 | 0.8613 |
| Старые тексты, метки answers.csv, без утечки | 0.9460 | 0.9407 |
| Текущие тексты, метки answers.csv, без утечки | 0.9141 | 0.9399 |

**Основное сравнение.** От старого способа без утечки (A1) к текущему (C) micro F1 вырос у обеих моделей. Для оценки разброса проведён парный bootstrap по группам query_cleaned: 2 000 повторов с одинаковыми выбранными группами для обоих вариантов.

| Модель | Разница micro F1, C − A1 | 95% bootstrap-интервал |
|---|---:|---:|
| Random Forest | +0.0461 | [0.0303; 0.0628] |
| Logistic Regression | +0.0786 | [0.0658; 0.0925] |

Оба интервала находятся выше нуля. На этой validation различие сохраняется при повторном выборе групп. Интервал условен для уже обученных моделей: он не учитывает неопределённость исходных меток, другие разбиения данных и вариативность повторного обучения.

**За счёт чего получено улучшение.** Основной прирост даёт замена словарных псевдометок на answers.csv (A1 → B). Изменение обработки текста (B → C) не улучшает все модели:

- Random Forest: разница micro F1 -0.0319, интервал [-0.0422; -0.0227].
- Logistic Regression: разница micro F1 -0.0008, интервал [-0.0060; +0.0044].

Для Random Forest вариант старой обработки текста с новой разметкой лучше текущего; у Logistic Regression уверенного различия обработки текста не выявлено. Это важное ограничение вывода: корректность обработки сама по себе не гарантирует более высокий F1. Исправление утечки обязательно для корректного протокола, даже если его влияние на число в конкретном эксперименте невелико.

**Что можно утверждать.** При одинаковых обучающих строках и одной validation текущий способ обучения лучше старого по согласованности с answers.csv. Эксперимент оценивает наличие категорий на выбранной validation. Validation используется для разработки, а test в этом эксперименте не использован. Параметры и рабочие Pipeline сохранены: вариант B можно рассмотреть в следующем эксперименте, без автоматической замены модели по одному результату. Исторические F1 ≈ 0,98 по псевдометкам с этими числами напрямую не сравниваются.


## Итог проекта

**Что сделано.** Построен воспроизводимый пайплайн классификации коротких запросов с несколькими метками: нормализация и лемматизация Natasha, словарные правила и две ML-модели на TF-IDF. Исходный текст сохраняется для проверки преобразований и сопоставления с разметкой.

**Что исправлено относительно учебной версии.** Целевые метки ML теперь берутся из answers.csv, а не создаются теми же словарными правилами. Разделение выполняется до обучения TF-IDF, совпадающие после очистки запросы остаются в одной части. В правилах сопоставляются целые слова и выражения. Baseline v2 дополнительно учитывает согласованные отрицания, варианты профессий, возраст, график и оплату.

**Почему прежний F1 ≈ 0,98 не доказывал качество.** Модель училась повторять метки собственного rule-based алгоритма; дополнительно TF-IDF обучался до train/test split. После изменения источника меток и протокола старые и новые числа напрямую не сравниваются.

**Сравнение на одном test.** Из 17 370 запросов 14 245 имеют строку в answers.csv. На размеченных данных получены train из 11 362 и test из 2 883 запросов. Остальные 3 125 запросов сохранены без метрик качества.

| Подход | Micro F1 | Macro F1 | Все пять меток совпали |
|---|---:|---:|---:|
| Словарный baseline v1 | 0.891 | 0.838 | 0.859 |
| Random Forest | 0.925 | 0.962 | 0.897 |
| Logistic Regression | 0.945 | 0.960 | 0.927 |

На этом split Logistic Regression имеет наибольшие micro F1 и долю полного совпадения меток; Random Forest немного выше по macro F1. Это сравнение согласия с существующей разметкой, а не подтверждённой точности относительно независимой экспертной оценки.

**Отдельный результат уточнения правил.** На 2 203 запросах validation разработки micro F1 baseline вырос с 0.899 до 0.916, macro F1 — с 0.848 до 0.906. Из validation исключены группы запросов ручного аудита. Эти значения нельзя напрямую сравнивать с таблицей test. Совпадение v2 на всех 80 примерах аудита — проверка согласованных правил на примерах их разработки. Baseline v2 на test не оценивался.

**Опыт сравнения обучения до и после изменений.** На одинаковых 9 116 обучающих и 2 203 validation-запросах текущий способ обучения превзошёл старые псевдометки и обработку без утечки: micro F1 Random Forest 0.868 → 0.914, Logistic Regression 0.861 → 0.940. Парные bootstrap-интервалы разницы находятся выше нуля. Главный вклад — замена источника меток; текущий вариант обработки текста не дал дополнительного выигрыша и ухудшил результат Random Forest. Это результат validation по answers.csv, а не новый test или независимая экспертная оценка.

**Проверка воспроизводимости.** 2026-10-10 все 40 кодовых ячеек, включая четыре ячейки сравнительного эксперимента, выполнены последовательно в новом ядре, с чтением CSV и обработкой текста с нуля. Метрики воспроизвелись; все 29 проверок правил прошли. Подтверждены отсутствие пересечения групп train/test и обучение TF-IDF только на train: 458 токенов, встречающихся только в test, не попали в его словари. Три CSV сохранены без изменений. Окружение: Python 3.10.20, pandas 2.3.3, Natasha 1.6.0, NLTK 3.10.3, scikit-learn 1.7.2; для совместимости pymorphy2 использовался setuptools 80.10.2.

**Ограничения и практический вывод.** Аудит выявил несоответствия разметки и различия критериев. Есть противоречивые метки внутри групп, проверен один split. Правила ограниченно понимают отрицания и вариативность языка. Метрики отражают наличие категорий, а не точность извлечения конкретной профессии. Проект показывает сравнение подходов, контроль утечки данных и анализ качества разметки; заявлять подтверждённую точность на новых запросах пока нельзя. Для такой оценки нужна отдельная проверенная выборка, не используемая для настройки.
